In [173]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [174]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..")
RAW_DATA = PROJECT_ROOT / "data" / "raw"

patients = pd.read_csv(RAW_DATA / "patients.csv")

print(f"Patients loaded: {len(patients):,}")
print(patients.head())

Patients loaded: 2,868
                                     Id   BIRTHDATE DEATHDATE          SSN  \
0  ebde6fa6-7f57-0893-8c60-9bc114b60899  2012-01-18       NaN  999-29-9099   
1  12021186-2496-a178-80b1-5c9e52adf9a3  2018-10-01       NaN  999-62-7777   
2  0b879291-4e25-a926-1253-0604f2f8913f  1973-09-06       NaN  999-77-2848   
3  229514f0-f87a-5e77-2a7b-c0112a2c5801  1992-07-20       NaN  999-23-5414   
4  05219dbb-95b5-7015-3172-52522117ea8a  1994-03-20       NaN  999-30-8427   

     DRIVERS    PASSPORT PREFIX        FIRST       MIDDLE          LAST  ...  \
0        NaN         NaN    NaN    Samuel331  Shanelle862  Bechtelar572  ...   
1        NaN         NaN    NaN      Elke785     Dolly486     Wisozk929  ...   
2  S99980595  X19589248X    Mr.  Ezequiel972      Mark765     Raynor401  ...   
3  S99939292  X80307275X    Ms.  Marcelle381      Dean966       Huel628  ...   
4  S99983683  X40068125X    Ms.   Michiko564   Chasity985     Russel238  ...   

            CITY          S

In [175]:
import pandas as pd
from pathlib import Path

# Project paths
PROJECT_ROOT = Path("..")
RAW_DATA = PROJECT_ROOT / "data" / "raw"

# Load raw Synthea datasets
patients = pd.read_csv(RAW_DATA / "patients.csv")
encounters = pd.read_csv(RAW_DATA / "encounters.csv")
claims = pd.read_csv(RAW_DATA / "claims.csv")
claims_transactions = pd.read_csv(
    RAW_DATA / "claims_transactions.csv"
)
providers = pd.read_csv(RAW_DATA / "providers.csv")
organizations = pd.read_csv(
    RAW_DATA / "organizations.csv"
)
payers = pd.read_csv(RAW_DATA / "payers.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [176]:
datasets = {
    "Patients": patients,
    "Encounters": encounters,
    "Claims": claims,
    "Claims Transactions": claims_transactions,
    "Providers": providers,
    "Organizations": organizations,
    "Payers": payers
}

overview = []

for name, df in datasets.items():
    overview.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate Rows": df.duplicated().sum(),
        "Missing Values": df.isna().sum().sum()
    })

overview_df = pd.DataFrame(overview)

overview_df

,Dataset,Rows,Columns,Duplicate Rows,Missing Values
0,Patients,2868,28,0,11315
1,Encounters,169640,15,0,124402
2,Claims,310474,31,0,2905068
3,Claims Transactions,2754409,33,0,24224398
4,Providers,1000,13,0,0
5,Organizations,1000,11,0,0
6,Payers,10,22,0,50


In [177]:
def missing_report(df, name):
    report = pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing %": (df.isna().mean() * 100).round(2),
        "Data Type": df.dtypes.astype(str)
    })

    report = report.sort_values(
        "Missing %",
        ascending=False
    )

    print(f"\n{name}")
    print("=" * 60)

    return report


patients_missing = missing_report(patients, "PATIENTS")

patients_missing


PATIENTS


,Missing Count,Missing %,Data Type
SUFFIX,2836,98.88,str
DEATHDATE,2500,87.17,str
MAIDEN,2091,72.91,str
MARITAL,930,32.43,str
FIPS,697,24.30,float64
PASSPORT,629,21.93,str
MIDDLE,574,20.01,str
PREFIX,566,19.74,str
DRIVERS,492,17.15,str
BIRTHDATE,0,0.00,str


In [178]:
# Create a copy of the raw Patients dataset
patients_clean = patients.copy()

# ---------------------------------------------------------
# 1. Remove columns that are not useful for analytics
# ---------------------------------------------------------

columns_to_drop = [
    "SUFFIX",
    "MAIDEN",
    "PASSPORT",
    "MIDDLE",
    "PREFIX",
    "DRIVERS"
]

patients_clean = patients_clean.drop(
    columns=columns_to_drop,
    errors="ignore"
)

# ---------------------------------------------------------
# 2. Convert dates
# ---------------------------------------------------------

patients_clean["BIRTHDATE"] = pd.to_datetime(
    patients_clean["BIRTHDATE"],
    errors="coerce"
)

patients_clean["DEATHDATE"] = pd.to_datetime(
    patients_clean["DEATHDATE"],
    errors="coerce"
)

# ---------------------------------------------------------
# 3. Create deceased indicator
# ---------------------------------------------------------

patients_clean["IsDeceased"] = (
    patients_clean["DEATHDATE"].notna()
)

# ---------------------------------------------------------
# 4. Handle missing marital status
# ---------------------------------------------------------

patients_clean["MARITAL"] = (
    patients_clean["MARITAL"]
    .fillna("Unknown")
)

# ---------------------------------------------------------
# 5. Calculate age
# ---------------------------------------------------------

reference_date = pd.Timestamp("2026-08-09")

patients_clean["Age"] = (
    (reference_date - patients_clean["BIRTHDATE"])
    .dt.days / 365.25
).round(1)

# ---------------------------------------------------------
# 6. Create age groups
# ---------------------------------------------------------

patients_clean["AgeGroup"] = pd.cut(
    patients_clean["Age"],
    bins=[0, 17, 34, 49, 64, 79, 120],
    labels=[
        "0-17",
        "18-34",
        "35-49",
        "50-64",
        "65-79",
        "80+"
    ],
    include_lowest=True
)

print("Patients cleaning completed.")
print(f"Rows: {len(patients_clean):,}")
print(f"Columns: {len(patients_clean.columns)}")

Patients cleaning completed.
Rows: 2,868
Columns: 25


In [179]:
patients_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2868 entries, 0 to 2867
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Id                   2868 non-null   str           
 1   BIRTHDATE            2868 non-null   datetime64[us]
 2   DEATHDATE            368 non-null    datetime64[us]
 3   SSN                  2868 non-null   str           
 4   FIRST                2868 non-null   str           
 5   LAST                 2868 non-null   str           
 6   MARITAL              2868 non-null   str           
 7   RACE                 2868 non-null   str           
 8   ETHNICITY            2868 non-null   str           
 9   GENDER               2868 non-null   str           
 10  BIRTHPLACE           2868 non-null   str           
 11  ADDRESS              2868 non-null   str           
 12  CITY                 2868 non-null   str           
 13  STATE                2868 non-null   str    

In [180]:
patients_clean[
    [
        "BIRTHDATE",
        "DEATHDATE",
        "IsDeceased",
        "Age",
        "AgeGroup",
        "MARITAL"
    ]
].head(10)

,BIRTHDATE,DEATHDATE,IsDeceased,Age,AgeGroup,MARITAL
0,2012-01-18,NaT,False,14.6,0-17,Unknown
1,2018-10-01,NaT,False,7.9,0-17,Unknown
2,1973-09-06,NaT,False,52.9,50-64,S
3,1992-07-20,NaT,False,34.1,35-49,S
4,1994-03-20,NaT,False,32.4,18-34,S
5,2003-01-13,NaT,False,23.6,18-34,Unknown
6,1986-12-11,NaT,False,39.7,35-49,M
7,1993-12-11,NaT,False,32.7,18-34,M
8,2020-02-20,NaT,False,6.5,0-17,Unknown
9,1999-07-05,NaT,False,27.1,18-34,Unknown


In [181]:
patients_clean["IsDeceased"].value_counts(dropna=False)

IsDeceased
False    2500
True      368
Name: count, dtype: int64

In [182]:
print("Age Validation")
print("=" * 40)

print("Negative ages:", (patients_clean["Age"] < 0).sum())
print("Age > 120:", (patients_clean["Age"] > 120).sum())
print("Missing age:", patients_clean["Age"].isna().sum())

Age Validation
Negative ages: 0
Age > 120: 0
Missing age: 0


In [183]:
# ============================================================
# Encounters - Initial Data Profiling
# ============================================================

print("Encounters Shape")
print("=" * 40)
print(f"Rows: {len(encounters):,}")
print(f"Columns: {len(encounters.columns)}")

print("\nColumn Information")
print("=" * 40)
print(encounters.dtypes)

print("\nMissing Values")
print("=" * 40)

encounters_missing = (
    encounters.isna()
    .sum()
    .sort_values(ascending=False)
)

encounters_missing_pct = (
    encounters.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

encounters_profile = pd.DataFrame({
    "Missing Count": encounters_missing,
    "Missing %": encounters_missing_pct.round(2),
    "Data Type": encounters.dtypes.astype(str)
})

encounters_profile

Encounters Shape
Rows: 169,640
Columns: 15

Column Information
Id                         str
START                      str
STOP                       str
PATIENT                    str
ORGANIZATION               str
PROVIDER                   str
PAYER                      str
ENCOUNTERCLASS             str
CODE                     int64
DESCRIPTION                str
BASE_ENCOUNTER_COST    float64
TOTAL_CLAIM_COST       float64
PAYER_COVERAGE         float64
REASONCODE             float64
REASONDESCRIPTION          str
dtype: object

Missing Values


,Missing Count,Missing %,Data Type
BASE_ENCOUNTER_COST,0,0.00,float64
CODE,0,0.00,int64
DESCRIPTION,0,0.00,str
ENCOUNTERCLASS,0,0.00,str
Id,0,0.00,str
ORGANIZATION,0,0.00,str
PATIENT,0,0.00,str
PAYER,0,0.00,str
PAYER_COVERAGE,0,0.00,float64
PROVIDER,0,0.00,str


In [184]:
print("Encounter Descriptions")
print("=" * 40)

print(
    encounters["DESCRIPTION"]
    .value_counts()
    .head(20)
)

Encounter Descriptions
DESCRIPTION
Encounter for problem (procedure)                                   45194
Encounter for check up (procedure)                                  26529
General examination of patient (procedure)                          25442
Encounter for symptom (procedure)                                    9515
Well child visit (procedure)                                         9219
Urgent care clinic (environment)                                     7205
Prenatal visit (regime/therapy)                                      6914
Patient encounter procedure (procedure)                              5282
Emergency room admission (procedure)                                 5273
Follow-up encounter (procedure)                                      4371
Outpatient procedure (procedure)                                     3659
Administration of vaccine to produce active immunity (procedure)     3314
Consultation for treatment (procedure)                               3210
Tel

In [185]:
print("Financial Validation")
print("=" * 40)

print("Negative BASE_ENCOUNTER_COST:",
      (encounters["BASE_ENCOUNTER_COST"] < 0).sum())

print("Negative TOTAL_CLAIM_COST:",
      (encounters["TOTAL_CLAIM_COST"] < 0).sum())

print("Negative PAYER_COVERAGE:",
      (encounters["PAYER_COVERAGE"] < 0).sum())

print("\nFinancial Summary")
print("=" * 40)

print(
    encounters[
        [
            "BASE_ENCOUNTER_COST",
            "TOTAL_CLAIM_COST",
            "PAYER_COVERAGE"
        ]
    ].describe()
)

Financial Validation
Negative BASE_ENCOUNTER_COST: 0
Negative TOTAL_CLAIM_COST: 0
Negative PAYER_COVERAGE: 0

Financial Summary
       BASE_ENCOUNTER_COST  TOTAL_CLAIM_COST  PAYER_COVERAGE
count        169640.000000     169640.000000   169640.000000
mean            112.373889       2715.965287     1973.354049
std              27.436532       6976.036205     5902.253250
min              75.000000         75.000000        0.000000
25%              85.550000        516.950000        0.000000
50%              87.710000        914.780000      485.870000
75%             142.580000       1871.780000     1303.700000
max             146.180000     244426.550000   207416.210000


In [186]:
encounters["START"] = pd.to_datetime(
    encounters["START"],
    errors="coerce"
)

encounters["STOP"] = pd.to_datetime(
    encounters["STOP"],
    errors="coerce"
)

print("Date Validation")
print("=" * 40)

print("Missing START:", encounters["START"].isna().sum())
print("Missing STOP:", encounters["STOP"].isna().sum())

print(
    "STOP before START:",
    (encounters["STOP"] < encounters["START"]).sum()
)

Date Validation
Missing START: 0
Missing STOP: 0
STOP before START: 0


In [187]:
# ============================================================
# Encounters - Data Cleaning
# ============================================================

encounters_clean = encounters.copy()

# ------------------------------------------------------------
# 1. Convert date/time fields
# ------------------------------------------------------------

encounters_clean["START"] = pd.to_datetime(
    encounters_clean["START"],
    errors="coerce"
)

encounters_clean["STOP"] = pd.to_datetime(
    encounters_clean["STOP"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2. Create encounter duration
# ------------------------------------------------------------

encounters_clean["EncounterDurationMinutes"] = (
    encounters_clean["STOP"] - encounters_clean["START"]
).dt.total_seconds() / 60

encounters_clean["EncounterDurationMinutes"] = (
    encounters_clean["EncounterDurationMinutes"].round(2)
)

# ------------------------------------------------------------
# 3. Create encounter date
# ------------------------------------------------------------

encounters_clean["EncounterDate"] = (
    encounters_clean["START"].dt.date
)

# ------------------------------------------------------------
# 4. Create useful time attributes
# ------------------------------------------------------------

encounters_clean["EncounterYear"] = (
    encounters_clean["START"].dt.year
)

encounters_clean["EncounterMonth"] = (
    encounters_clean["START"].dt.month
)

encounters_clean["EncounterMonthName"] = (
    encounters_clean["START"].dt.month_name()
)

# ------------------------------------------------------------
# 5. Flag encounters with a documented reason
# ------------------------------------------------------------

encounters_clean["HasReason"] = (
    encounters_clean["REASONCODE"].notna()
)

# ------------------------------------------------------------
# 6. Clean text fields
# ------------------------------------------------------------

text_columns = [
    "ENCOUNTERCLASS",
    "DESCRIPTION",
    "REASONDESCRIPTION"
]

for column in text_columns:
    encounters_clean[column] = (
        encounters_clean[column]
        .str.strip()
    )

# ------------------------------------------------------------
# 7. Preserve missing reasons
# ------------------------------------------------------------
# We do NOT invent a medical reason where one is missing.
# Missing reason remains null.

print("Encounters cleaning completed.")
print(f"Rows: {len(encounters_clean):,}")
print(f"Columns: {len(encounters_clean.columns)}")

Encounters cleaning completed.
Rows: 169,640
Columns: 21


In [188]:
print("Encounter Duration Validation")
print("=" * 40)

print(
    "Negative durations:",
    (encounters_clean["EncounterDurationMinutes"] < 0).sum()
)

print(
    "Missing durations:",
    encounters_clean["EncounterDurationMinutes"].isna().sum()
)

print(
    "Zero-minute encounters:",
    (encounters_clean["EncounterDurationMinutes"] == 0).sum()
)

print("\nDuration Summary")
print("=" * 40)

print(
    encounters_clean["EncounterDurationMinutes"].describe()
)

Encounter Duration Validation
Negative durations: 0
Missing durations: 0
Zero-minute encounters: 0

Duration Summary
count    169640.000000
mean        340.888340
std        2926.713088
min          15.000000
25%          15.000000
50%          45.530000
75%         147.000000
max      165615.000000
Name: EncounterDurationMinutes, dtype: float64


In [189]:
print("Financial Validation")
print("=" * 40)

print(
    "Total encounter cost:",
    encounters_clean["TOTAL_CLAIM_COST"].sum()
)

print(
    "Total payer coverage:",
    encounters_clean["PAYER_COVERAGE"].sum()
)

print(
    "Average encounter cost:",
    encounters_clean["TOTAL_CLAIM_COST"].mean()
)

print(
    "Maximum encounter cost:",
    encounters_clean["TOTAL_CLAIM_COST"].max()
)

Financial Validation
Total encounter cost: 460736351.21000004
Total payer coverage: 334759780.82
Average encounter cost: 2715.9652865479843
Maximum encounter cost: 244426.55


In [190]:
print(
    encounters_clean["ENCOUNTERCLASS"]
    .value_counts(dropna=False)
)

ENCOUNTERCLASS
ambulatory    94578
wellness      35021
outpatient    21317
urgentcare     7243
emergency      6805
inpatient      2711
home            681
snf             451
virtual         429
hospice         404
Name: count, dtype: int64


In [191]:
# ============================================================
# Standardize Encounter Categories
# ============================================================

encounter_category_map = {
    "ambulatory": "Outpatient",
    "wellness": "Preventive",
    "outpatient": "Outpatient",
    "urgentcare": "Urgent Care",
    "emergency": "Emergency",
    "inpatient": "Inpatient",
    "home": "Home Care",
    "snf": "Skilled Nursing",
    "virtual": "Telehealth",
    "hospice": "Hospice"
}

encounters_clean["EncounterCategory"] = (
    encounters_clean["ENCOUNTERCLASS"]
    .map(encounter_category_map)
)

print("Encounter Category Validation")
print("=" * 40)

print(
    encounters_clean["EncounterCategory"]
    .value_counts(dropna=False)
)

Encounter Category Validation
EncounterCategory
Outpatient         115895
Preventive          35021
Urgent Care          7243
Emergency            6805
Inpatient            2711
Home Care             681
Skilled Nursing       451
Telehealth            429
Hospice               404
Name: count, dtype: int64


In [192]:
unmapped = encounters_clean[
    encounters_clean["EncounterCategory"].isna()
]["ENCOUNTERCLASS"].unique()

print("Unmapped encounter classes:")
print(unmapped)

Unmapped encounter classes:
<StringArray>
[]
Length: 0, dtype: str


In [193]:
# ============================================================
# Financial Metrics
# ============================================================

# Patient responsibility
encounters_clean["PatientResponsibility"] = (
    encounters_clean["TOTAL_CLAIM_COST"]
    - encounters_clean["PAYER_COVERAGE"]
)

# Percentage of claim covered by payer
encounters_clean["PayerCoverageRate"] = (
    encounters_clean["PAYER_COVERAGE"]
    / encounters_clean["TOTAL_CLAIM_COST"]
)

# Prevent division by zero
encounters_clean["PayerCoverageRate"] = (
    encounters_clean["PayerCoverageRate"]
    .replace([float("inf"), -float("inf")], pd.NA)
)

# Round financial metrics
encounters_clean["PatientResponsibility"] = (
    encounters_clean["PatientResponsibility"].round(2)
)

encounters_clean["PayerCoverageRate"] = (
    encounters_clean["PayerCoverageRate"].round(4)
)

print("Financial metrics created.")

Financial metrics created.


In [194]:
print("Financial Metrics Validation")
print("=" * 40)

print(
    "Negative patient responsibility:",
    (encounters_clean["PatientResponsibility"] < 0).sum()
)

print(
    "Missing payer coverage rate:",
    encounters_clean["PayerCoverageRate"].isna().sum()
)

print("\nFinancial Metrics Summary")
print("=" * 40)

print(
    encounters_clean[
        [
            "TOTAL_CLAIM_COST",
            "PAYER_COVERAGE",
            "PatientResponsibility",
            "PayerCoverageRate"
        ]
    ].describe()
)

Financial Metrics Validation
Negative patient responsibility: 0
Missing payer coverage rate: 0

Financial Metrics Summary
       TOTAL_CLAIM_COST  PAYER_COVERAGE  PatientResponsibility  \
count     169640.000000   169640.000000          169640.000000   
mean        2715.965287     1973.354049             742.611238   
std         6976.036205     5902.253250            2404.846038   
min           75.000000        0.000000               0.000000   
25%          516.950000        0.000000              50.000000   
50%          914.780000      485.870000             144.640000   
75%         1871.780000     1303.700000             704.200000   
max       244426.550000   207416.210000          173414.630000   

       PayerCoverageRate  
count      169640.000000  
mean            0.608348  
std             0.389672  
min             0.000000  
25%             0.000000  
50%             0.800000  
75%             0.906700  
max             1.000000  


In [195]:
from pathlib import Path

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Processed folder: {PROCESSED_DATA}")

Processed folder: ..\data\processed


In [196]:
encounters_clean.to_csv(
    PROCESSED_DATA / "encounters_clean.csv",
    index=False
)

print("encounters_clean.csv exported successfully.")

encounters_clean.csv exported successfully.


In [197]:
print(
    f"Rows exported: {len(encounters_clean):,}"
)

print(
    f"Columns exported: {len(encounters_clean.columns)}"
)

print(
    "File exists:",
    (PROCESSED_DATA / "encounters_clean.csv").exists()
)

Rows exported: 169,640
Columns exported: 24
File exists: True


In [198]:
# ============================================================
# Claims - Initial Data Profiling
# ============================================================

print("Claims Shape")
print("=" * 40)

print(f"Rows: {len(claims):,}")
print(f"Columns: {len(claims.columns)}")

print("\nColumn Information")
print("=" * 40)

print(claims.dtypes)

print("\nMissing Values")
print("=" * 40)

claims_missing = (
    claims.isna()
    .sum()
    .sort_values(ascending=False)
)

claims_missing_pct = (
    claims.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

claims_profile = pd.DataFrame({
    "Missing Count": claims_missing,
    "Missing %": claims_missing_pct.round(2),
    "Data Type": claims.dtypes.astype(str)
})

claims_profile

Claims Shape
Rows: 310,474
Columns: 31

Column Information
Id                                 str
PATIENTID                          str
PROVIDERID                         str
PRIMARYPATIENTINSURANCEID          str
SECONDARYPATIENTINSURANCEID        str
DEPARTMENTID                     int64
PATIENTDEPARTMENTID              int64
DIAGNOSIS1                       int64
DIAGNOSIS2                     float64
DIAGNOSIS3                     float64
DIAGNOSIS4                     float64
DIAGNOSIS5                     float64
DIAGNOSIS6                     float64
DIAGNOSIS7                     float64
DIAGNOSIS8                     float64
REFERRINGPROVIDERID            float64
APPOINTMENTID                      str
CURRENTILLNESSDATE                 str
SERVICEDATE                        str
SUPERVISINGPROVIDERID              str
STATUS1                            str
STATUS2                            str
STATUSP                            str
OUTSTANDING1                   float64
OUTST

,Missing Count,Missing %,Data Type
APPOINTMENTID,0,0.00,str
CURRENTILLNESSDATE,0,0.00,str
DEPARTMENTID,0,0.00,int64
DIAGNOSIS1,0,0.00,int64
DIAGNOSIS2,220874,71.14,float64
DIAGNOSIS3,280059,90.20,float64
DIAGNOSIS4,301966,97.26,float64
DIAGNOSIS5,307686,99.10,float64
DIAGNOSIS6,309308,99.62,float64
DIAGNOSIS7,310132,99.89,float64


In [199]:
print("Claim Status Values")
print("=" * 40)

print(
    claims[
        [
            "STATUS1",
            "STATUS2",
            "STATUSP"
        ]
    ].value_counts(dropna=False).head(20)
)

Claim Status Values
STATUS1  STATUS2  STATUSP
CLOSED   CLOSED   CLOSED     199532
         NaN      CLOSED     110581
BILLED   NaN      BILLED        196
         BILLED   BILLED        165
Name: count, dtype: int64


In [200]:
print("Claim Financial Fields")
print("=" * 40)

print(
    claims[
        [
            "OUTSTANDING1",
            "OUTSTANDING2",
            "OUTSTANDINGP"
        ]
    ].describe()
)

Claim Financial Fields
        OUTSTANDING1   OUTSTANDING2   OUTSTANDINGP
count  310474.000000  199697.000000  310474.000000
mean        0.426525       0.209690       0.132528
std        48.352138      31.229736      12.924954
min         0.000000       0.000000       0.000000
25%         0.000000       0.000000       0.000000
50%         0.000000       0.000000       0.000000
75%         0.000000       0.000000       0.000000
max     11303.930000   11303.930000    3437.190000


In [201]:
# ============================================================
# Claims - Key Validation
# ============================================================

print("Claims Key Validation")
print("=" * 40)

print(
    "Duplicate claim IDs:",
    claims["Id"].duplicated().sum()
)

print(
    "Duplicate patient IDs:",
    claims["PATIENTID"].duplicated().sum()
)

print(
    "Unique claims:",
    claims["Id"].nunique()
)

print(
    "Unique patients:",
    claims["PATIENTID"].nunique()
)

Claims Key Validation
Duplicate claim IDs: 0
Duplicate patient IDs: 307606
Unique claims: 310474
Unique patients: 2868


In [202]:
# ============================================================
# Claims - Date Validation
# ============================================================

claims["CURRENTILLNESSDATE"] = pd.to_datetime(
    claims["CURRENTILLNESSDATE"],
    errors="coerce"
)

claims["SERVICEDATE"] = pd.to_datetime(
    claims["SERVICEDATE"],
    errors="coerce"
)

print("Claim Date Validation")
print("=" * 40)

print(
    "Missing CURRENTILLNESSDATE:",
    claims["CURRENTILLNESSDATE"].isna().sum()
)

print(
    "Missing SERVICEDATE:",
    claims["SERVICEDATE"].isna().sum()
)

print(
    "Illness date after service date:",
    (
        claims["CURRENTILLNESSDATE"]
        > claims["SERVICEDATE"]
    ).sum()
)

Claim Date Validation
Missing CURRENTILLNESSDATE: 0
Missing SERVICEDATE: 0
Illness date after service date: 0


In [203]:
# ============================================================
# Claims - Outstanding Balance Analysis
# ============================================================

outstanding_columns = [
    "OUTSTANDING1",
    "OUTSTANDING2",
    "OUTSTANDINGP"
]

print("Outstanding Balance Analysis")
print("=" * 40)

for column in outstanding_columns:
    print(
        f"{column} > $0:",
        (claims[column].fillna(0) > 0).sum()
    )

print("\nTotal Outstanding by Field")
print("=" * 40)

for column in outstanding_columns:
    print(
        f"{column}: "
        f"${claims[column].fillna(0).sum():,.2f}"
    )

Outstanding Balance Analysis
OUTSTANDING1 > $0: 310
OUTSTANDING2 > $0: 124
OUTSTANDINGP > $0: 305

Total Outstanding by Field
OUTSTANDING1: $132,424.95
OUTSTANDING2: $41,874.43
OUTSTANDINGP: $41,146.38


In [204]:
claims["TotalOutstanding"] = (
    claims["OUTSTANDING1"].fillna(0)
    + claims["OUTSTANDING2"].fillna(0)
    + claims["OUTSTANDINGP"].fillna(0)
)

print(
    "Total outstanding:",
    f"${claims['TotalOutstanding'].sum():,.2f}"
)

Total outstanding: $215,445.76


In [205]:
# ============================================================
# Claims - Data Cleaning
# ============================================================

claims_clean = claims.copy()

# ------------------------------------------------------------
# 1. Clean date fields
# ------------------------------------------------------------

claims_clean["CURRENTILLNESSDATE"] = pd.to_datetime(
    claims_clean["CURRENTILLNESSDATE"],
    errors="coerce"
)

claims_clean["SERVICEDATE"] = pd.to_datetime(
    claims_clean["SERVICEDATE"],
    errors="coerce"
)

claims_clean["LASTBILLEDDATE1"] = pd.to_datetime(
    claims_clean["LASTBILLEDDATE1"],
    errors="coerce"
)

claims_clean["LASTBILLEDDATE2"] = pd.to_datetime(
    claims_clean["LASTBILLEDDATE2"],
    errors="coerce"
)

claims_clean["LASTBILLEDDATEP"] = pd.to_datetime(
    claims_clean["LASTBILLEDDATEP"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2. Clean claim status fields
# ------------------------------------------------------------

status_columns = [
    "STATUS1",
    "STATUS2",
    "STATUSP"
]

for column in status_columns:
    claims_clean[column] = (
        claims_clean[column]
        .str.strip()
        .str.upper()
    )

# ------------------------------------------------------------
# 3. Create total outstanding balance
# ------------------------------------------------------------

claims_clean["TotalOutstanding"] = (
    claims_clean["OUTSTANDING1"].fillna(0)
    + claims_clean["OUTSTANDING2"].fillna(0)
    + claims_clean["OUTSTANDINGP"].fillna(0)
)

claims_clean["TotalOutstanding"] = (
    claims_clean["TotalOutstanding"].round(2)
)

# ------------------------------------------------------------
# 4. Create outstanding flag
# ------------------------------------------------------------

claims_clean["HasOutstandingBalance"] = (
    claims_clean["TotalOutstanding"] > 0
)

# ------------------------------------------------------------
# 5. Create service year/month
# ------------------------------------------------------------

claims_clean["ServiceYear"] = (
    claims_clean["SERVICEDATE"].dt.year
)

claims_clean["ServiceMonth"] = (
    claims_clean["SERVICEDATE"].dt.month
)

claims_clean["ServiceMonthName"] = (
    claims_clean["SERVICEDATE"].dt.month_name()
)

print("Claims cleaning completed.")
print(f"Rows: {len(claims_clean):,}")
print(f"Columns: {len(claims_clean.columns)}")

Claims cleaning completed.
Rows: 310,474
Columns: 36


In [206]:
print("Claims Financial Validation")
print("=" * 40)

print(
    "Negative TotalOutstanding:",
    (claims_clean["TotalOutstanding"] < 0).sum()
)

print(
    "Claims with outstanding balance:",
    claims_clean["HasOutstandingBalance"].sum()
)

print(
    "Total outstanding:",
    f"${claims_clean['TotalOutstanding'].sum():,.2f}"
)

Claims Financial Validation
Negative TotalOutstanding: 0
Claims with outstanding balance: 361
Total outstanding: $215,445.76


In [207]:
claims_clean.to_csv(
    PROCESSED_DATA / "claims_clean.csv",
    index=False
)

print("claims_clean.csv exported successfully.")

claims_clean.csv exported successfully.


In [208]:
print(
    "File exists:",
    (PROCESSED_DATA / "claims_clean.csv").exists()
)

File exists: True


In [209]:
# ============================================================
# 05. CLAIMS TRANSACTIONS PROFILING
# ============================================================

In [210]:

claims_transactions.shape

(2754409, 33)

In [211]:
claims_transactions.head()

,ID,CLAIMID,CHARGEID,PATIENTID,TYPE,AMOUNT,METHOD,FROMDATE,TODATE,PLACEOFSERVICE,...,PAYMENTS,ADJUSTMENTS,TRANSFERS,OUTSTANDING,APPOINTMENTID,LINENOTE,PATIENTINSURANCEID,FEESCHEDULEID,PROVIDERID,SUPERVISINGPROVIDERID
0,d3098578-1e29-69a3-9289-b399ffa5219b,12021186-2496-a178-a177-004c12c64622,1,12021186-2496-a178-80b1-5c9e52adf9a3,CHARGE,136.80,NaN,2018-10-01T14:53:56Z,2018-10-01T15:08:56Z,b574fc4f-313c-38a4-9baf-7e12e431a499,...,0.0,0.0,NaN,0.0,12021186-2496-a178-c232-3afa8dcfd265,NaN,12021186-2496-a178-94d8-5d4822d90438,1,44f18836-01b5-3101-beca-390e3e1bb350,44f18836-01b5-3101-beca-390e3e1bb350
1,c39ff696-95bf-62a2-0b26-d7881f551412,0b879291-4e25-a926-f6f8-e41f2bf6a9a6,2,0b879291-4e25-a926-1253-0604f2f8913f,CHARGE,136.80,NaN,1991-10-31T14:16:01Z,1991-10-31T14:58:35Z,31e00f85-6909-3d9f-afc6-5ae507ab0c06,...,0.0,0.0,NaN,0.0,0b879291-4e25-a926-4097-793c8006d6bc,NaN,NaN,1,13b552ae-997c-33c8-9f0f-88ac322ba107,13b552ae-997c-33c8-9f0f-88ac322ba107
2,4c2615e0-78c3-a68f-4a26-ce67feb2866e,12021186-2496-a178-a177-004c12c64622,3,12021186-2496-a178-80b1-5c9e52adf9a3,TRANSFEROUT,NaN,NaN,2018-10-01T14:53:56Z,2018-10-01T15:08:56Z,b574fc4f-313c-38a4-9baf-7e12e431a499,...,0.0,0.0,136.8,136.8,12021186-2496-a178-c232-3afa8dcfd265,NaN,12021186-2496-a178-94d8-5d4822d90438,1,44f18836-01b5-3101-beca-390e3e1bb350,44f18836-01b5-3101-beca-390e3e1bb350
3,e6a84b1f-4143-f3ab-18db-a808525e3630,ebde6fa6-7f57-0893-253f-1f8dcefcd8d6,0,ebde6fa6-7f57-0893-8c60-9bc114b60899,CHARGE,85.55,NaN,2013-09-28T22:10:35Z,2013-09-28T22:25:35Z,6aae7a31-90df-3455-ad8d-81f8cf2d21e8,...,0.0,0.0,NaN,0.0,ebde6fa6-7f57-0893-748d-6eeeccfa2731,NaN,ebde6fa6-7f57-0893-5a51-d12db313bbf4,1,b246e387-68c1-36e8-9991-9be11f671291,b246e387-68c1-36e8-9991-9be11f671291
4,30985781-e39e-9a3d-289b-399ffb5a19b9,12021186-2496-a178-a177-004c12c64622,5,12021186-2496-a178-80b1-5c9e52adf9a3,TRANSFERIN,136.80,NaN,2018-10-01T14:53:56Z,2018-10-01T15:08:56Z,b574fc4f-313c-38a4-9baf-7e12e431a499,...,0.0,0.0,136.8,136.8,12021186-2496-a178-c232-3afa8dcfd265,NaN,12021186-2496-a178-94d8-5d4822d90438,1,44f18836-01b5-3101-beca-390e3e1bb350,44f18836-01b5-3101-beca-390e3e1bb350


In [212]:
claims_transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 2754409 entries, 0 to 2754408
Data columns (total 33 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ID                     str    
 1   CLAIMID                str    
 2   CHARGEID               int64  
 3   PATIENTID              str    
 4   TYPE                   str    
 5   AMOUNT                 float64
 6   METHOD                 str    
 7   FROMDATE               str    
 8   TODATE                 str    
 9   PLACEOFSERVICE         str    
 10  PROCEDURECODE          int64  
 11  MODIFIER1              float64
 12  MODIFIER2              float64
 13  DIAGNOSISREF1          int64  
 14  DIAGNOSISREF2          float64
 15  DIAGNOSISREF3          float64
 16  DIAGNOSISREF4          float64
 17  UNITS                  int64  
 18  DEPARTMENTID           int64  
 19  NOTES                  str    
 20  UNITAMOUNT             float64
 21  TRANSFEROUTID          float64
 22  TRANSFERTYPE           str   

In [213]:
print("Claims Transactions Profile")
print("=" * 50)

print(f"Rows: {len(claims_transactions):,}")
print(f"Columns: {len(claims_transactions.columns)}")

print("\nDuplicate Rows:")
print(claims_transactions.duplicated().sum())

print("\nDuplicate Transaction IDs:")
print(
    claims_transactions["ID"].duplicated().sum()
)

print("\nDuplicate Claim IDs:")
print(
    claims_transactions["CLAIMID"].duplicated().sum()
)

print("\nTransaction Types:")
print(
    claims_transactions["TYPE"].value_counts(dropna=False)
)

Claims Transactions Profile
Rows: 2,754,409
Columns: 33

Duplicate Rows:
0

Duplicate Transaction IDs:
0

Duplicate Claim IDs:
2443935

Transaction Types:
TYPE
PAYMENT        1043063
CHARGE          811884
TRANSFEROUT     449731
TRANSFERIN      449731
Name: count, dtype: int64


In [214]:
# ============================================================
# Claims Transactions - Financial Profiling
# ============================================================

financial_columns = [
    "AMOUNT",
    "UNITAMOUNT",
    "PAYMENTS",
    "ADJUSTMENTS",
    "TRANSFERS",
    "OUTSTANDING"
]

print("Financial Fields Summary")
print("=" * 50)

print(
    claims_transactions[financial_columns]
    .describe()
    .T
)

Financial Fields Summary
                 count        mean          std   min    25%     50%     75%  \
AMOUNT       1261615.0  414.487673  1531.003994  0.00  29.21  129.94  431.40   
UNITAMOUNT   2754409.0  158.695089  1035.429831  0.00   0.00    0.00    0.00   
PAYMENTS     2754409.0  158.695089   935.578025  0.00   0.00    0.00   85.55   
ADJUSTMENTS  2754409.0    0.000000     0.000000  0.00   0.00    0.00    0.00   
TRANSFERS     899462.0  190.808914   548.194511  0.01  27.20   86.28  136.80   
OUTSTANDING  2754409.0   83.027109   415.400367  0.00   0.00    0.00   86.28   

                   max  
AMOUNT       161615.00  
UNITAMOUNT   161615.00  
PAYMENTS     129292.00  
ADJUSTMENTS       0.00  
TRANSFERS     42306.68  
OUTSTANDING  103923.40  


In [215]:
print("Missing Financial Values")
print("=" * 50)

print(
    claims_transactions[financial_columns]
    .isna()
    .sum()
)

Missing Financial Values
AMOUNT         1492794
UNITAMOUNT           0
PAYMENTS             0
ADJUSTMENTS          0
TRANSFERS      1854947
OUTSTANDING          0
dtype: int64


In [216]:
print("Negative Financial Values")
print("=" * 50)

for column in financial_columns:
    print(
        f"{column}:",
        (claims_transactions[column] < 0).sum()
    )

Negative Financial Values
AMOUNT: 0
UNITAMOUNT: 0
PAYMENTS: 0
ADJUSTMENTS: 0
TRANSFERS: 0
OUTSTANDING: 0


In [217]:
print("Transaction Type Financial Summary")
print("=" * 50)

print(
    claims_transactions
    .groupby("TYPE")[
        ["AMOUNT", "PAYMENTS", "ADJUSTMENTS", "TRANSFERS", "OUTSTANDING"]
    ]
    .agg(["count", "sum", "mean"])
)

Transaction Type Financial Summary
             AMOUNT                           PAYMENTS                \
              count           sum        mean    count           sum   
TYPE                                                                   
CHARGE       811884  4.371112e+08  538.391177   811884  0.000000e+00   
PAYMENT           0  0.000000e+00         NaN  1043063  4.371112e+08   
TRANSFERIN   449731  8.581268e+07  190.808914   449731  0.000000e+00   
TRANSFEROUT       0  0.000000e+00         NaN   449731  0.000000e+00   

                        ADJUSTMENTS           TRANSFERS               \
                   mean       count  sum mean     count          sum   
TYPE                                                                   
CHARGE         0.000000      811884  0.0  0.0         0         0.00   
PAYMENT      419.064987     1043063  0.0  0.0         0         0.00   
TRANSFERIN     0.000000      449731  0.0  0.0    449731  85812683.56   
TRANSFEROUT    0.000000     

In [218]:
# ============================================================
# Claims Transactions - Missing Value Analysis
# ============================================================

financial_columns = [
    "AMOUNT",
    "UNITAMOUNT",
    "PAYMENTS",
    "ADJUSTMENTS",
    "TRANSFERS",
    "OUTSTANDING"
]

missing_financial = pd.DataFrame({
    "Missing Count": claims_transactions[financial_columns].isna().sum(),
    "Missing %": (
        claims_transactions[financial_columns].isna().mean() * 100
    ).round(2)
})

missing_financial.sort_values(
    "Missing %",
    ascending=False
)

,Missing Count,Missing %
TRANSFERS,1854947,67.34
AMOUNT,1492794,54.20
UNITAMOUNT,0,0.00
PAYMENTS,0,0.00
ADJUSTMENTS,0,0.00
OUTSTANDING,0,0.00


In [219]:
# ============================================================
# Missing Financial Fields by Transaction Type
# ============================================================

for column in financial_columns:
    print(f"\n{column}")
    print("-" * 30)

    print(
        claims_transactions
        .groupby("TYPE")[column]
        .apply(lambda x: x.isna().sum())
    )


AMOUNT
------------------------------
TYPE
CHARGE               0
PAYMENT        1043063
TRANSFERIN           0
TRANSFEROUT     449731
Name: AMOUNT, dtype: int64

UNITAMOUNT
------------------------------
TYPE
CHARGE         0
PAYMENT        0
TRANSFERIN     0
TRANSFEROUT    0
Name: UNITAMOUNT, dtype: int64

PAYMENTS
------------------------------
TYPE
CHARGE         0
PAYMENT        0
TRANSFERIN     0
TRANSFEROUT    0
Name: PAYMENTS, dtype: int64

ADJUSTMENTS
------------------------------
TYPE
CHARGE         0
PAYMENT        0
TRANSFERIN     0
TRANSFEROUT    0
Name: ADJUSTMENTS, dtype: int64

TRANSFERS
------------------------------
TYPE
CHARGE          811884
PAYMENT        1043063
TRANSFERIN           0
TRANSFEROUT          0
Name: TRANSFERS, dtype: int64

OUTSTANDING
------------------------------
TYPE
CHARGE         0
PAYMENT        0
TRANSFERIN     0
TRANSFEROUT    0
Name: OUTSTANDING, dtype: int64


In [220]:
# ============================================================
# Claims Transactions - Negative Value Validation
# ============================================================

print("Negative Financial Values")
print("=" * 50)

for column in financial_columns:
    print(
        f"{column}:",
        (claims_transactions[column].fillna(0) < 0).sum()
    )

Negative Financial Values
AMOUNT: 0
UNITAMOUNT: 0
PAYMENTS: 0
ADJUSTMENTS: 0
TRANSFERS: 0
OUTSTANDING: 0


In [221]:
# ============================================================
# Financial Totals by Transaction Type
# ============================================================

financial_totals = (
    claims_transactions
    .groupby("TYPE")
    .agg(
        TransactionCount=("ID", "count"),
        TotalAmount=("AMOUNT", "sum"),
        TotalPayments=("PAYMENTS", "sum"),
        TotalAdjustments=("ADJUSTMENTS", "sum"),
        TotalTransfers=("TRANSFERS", "sum"),
        TotalOutstanding=("OUTSTANDING", "sum")
    )
    .round(2)
)

financial_totals

,TransactionCount,TotalAmount,TotalPayments,TotalAdjustments,TotalTransfers,TotalOutstanding
TYPE,,,,,,
CHARGE,811884,4.371112e+08,0.000000e+00,0.0,0.00,0.00
PAYMENT,1043063,0.000000e+00,4.371112e+08,0.0,0.00,57065249.20
TRANSFERIN,449731,8.581268e+07,0.000000e+00,0.0,85812683.56,85812683.56
TRANSFEROUT,449731,0.000000e+00,0.000000e+00,0.0,85812683.56,85812683.56


In [222]:
# ============================================================
# Claims Transactions - Cleaning & Transformation
# ============================================================

claims_transactions_clean = claims_transactions.copy()

# ------------------------------------------------------------
# 1. Clean text fields
# ------------------------------------------------------------

text_columns = [
    "TYPE",
    "METHOD",
    "PLACEOFSERVICE",
    "NOTES",
    "LINENOTE"
]

for column in text_columns:
    if column in claims_transactions_clean.columns:
        claims_transactions_clean[column] = (
            claims_transactions_clean[column]
            .astype("string")
            .str.strip()
        )

# Normalize transaction type
claims_transactions_clean["TYPE"] = (
    claims_transactions_clean["TYPE"]
    .str.upper()
)

# ------------------------------------------------------------
# 2. Convert financial fields to numeric
# ------------------------------------------------------------

financial_columns = [
    "AMOUNT",
    "UNITAMOUNT",
    "PAYMENTS",
    "ADJUSTMENTS",
    "TRANSFERS",
    "OUTSTANDING"
]

for column in financial_columns:
    claims_transactions_clean[column] = pd.to_numeric(
        claims_transactions_clean[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# 3. Replace missing financial values with 0
# ------------------------------------------------------------
# For transaction-level analysis, a missing financial
# component means no amount was recorded for that component.

for column in financial_columns:
    claims_transactions_clean[column] = (
        claims_transactions_clean[column].fillna(0)
    )

# ------------------------------------------------------------
# 4. Round financial values
# ------------------------------------------------------------

for column in financial_columns:
    claims_transactions_clean[column] = (
        claims_transactions_clean[column].round(2)
    )

# ------------------------------------------------------------
# 5. Create transaction categories
# ------------------------------------------------------------

claims_transactions_clean["TransactionCategory"] = (
    claims_transactions_clean["TYPE"].map({
        "CHARGE": "Charge",
        "PAYMENT": "Payment",
        "TRANSFERIN": "Transfer In",
        "TRANSFEROUT": "Transfer Out"
    })
)

# ------------------------------------------------------------
# 6. Create financial classification
# ------------------------------------------------------------

claims_transactions_clean["FinancialFlow"] = (
    claims_transactions_clean["TYPE"].map({
        "CHARGE": "Billed",
        "PAYMENT": "Collected",
        "TRANSFERIN": "Transferred In",
        "TRANSFEROUT": "Transferred Out"
    })
)

print("Claims Transactions cleaning completed.")
print(f"Rows: {len(claims_transactions_clean):,}")
print(f"Columns: {len(claims_transactions_clean.columns)}")

Claims Transactions cleaning completed.
Rows: 2,754,409
Columns: 35


In [223]:
# ============================================================
# Claims Transactions - Financial Indicators
# ============================================================

claims_transactions_clean["ChargeAmount"] = (
    claims_transactions_clean["AMOUNT"]
    .where(
        claims_transactions_clean["TYPE"] == "CHARGE",
        0
    )
)

claims_transactions_clean["PaymentAmount"] = (
    claims_transactions_clean["PAYMENTS"]
    .where(
        claims_transactions_clean["TYPE"] == "PAYMENT",
        0
    )
)

claims_transactions_clean["TransferAmount"] = (
    claims_transactions_clean["TRANSFERS"]
    .where(
        claims_transactions_clean["TYPE"].isin(
            ["TRANSFERIN", "TRANSFEROUT"]
        ),
        0
    )
)

claims_transactions_clean["OutstandingAmount"] = (
    claims_transactions_clean["OUTSTANDING"]
)

print("Financial indicators created.")

Financial indicators created.


In [224]:
# ============================================================
# Claims Transactions - Final Financial Validation
# ============================================================

print("FINAL FINANCIAL VALIDATION")
print("=" * 50)

print(
    "Total Charges:",
    f"${claims_transactions_clean['ChargeAmount'].sum():,.2f}"
)

print(
    "Total Payments:",
    f"${claims_transactions_clean['PaymentAmount'].sum():,.2f}"
)

print(
    "Total Transfers:",
    f"${claims_transactions_clean['TransferAmount'].sum():,.2f}"
)

print(
    "Total Outstanding:",
    f"${claims_transactions_clean['OutstandingAmount'].sum():,.2f}"
)

print("\nTransaction Categories:")
print(
    claims_transactions_clean["TransactionCategory"]
    .value_counts(dropna=False)
)

FINAL FINANCIAL VALIDATION
Total Charges: $437,111,182.35
Total Payments: $437,111,182.35
Total Transfers: $171,625,367.12
Total Outstanding: $228,690,616.32

Transaction Categories:
TransactionCategory
Payment         1043063
Charge           811884
Transfer Out     449731
Transfer In      449731
Name: count, dtype: int64


In [225]:
# ============================================================
# Claims Transactions - Financial Flow Refinement
# ============================================================

claims_transactions_clean["TransferInAmount"] = (
    claims_transactions_clean["TRANSFERS"]
    .where(
        claims_transactions_clean["TYPE"] == "TRANSFERIN",
        0
    )
)

claims_transactions_clean["TransferOutAmount"] = (
    claims_transactions_clean["TRANSFERS"]
    .where(
        claims_transactions_clean["TYPE"] == "TRANSFEROUT",
        0
    )
)

# Financial balance movement
claims_transactions_clean["NetTransferAmount"] = (
    claims_transactions_clean["TransferInAmount"]
    - claims_transactions_clean["TransferOutAmount"]
)

print("Transfer fields created.")

Transfer fields created.


In [226]:
print("Transfer Validation")
print("=" * 40)

print(
    "Transfer In:",
    f"${claims_transactions_clean['TransferInAmount'].sum():,.2f}"
)

print(
    "Transfer Out:",
    f"${claims_transactions_clean['TransferOutAmount'].sum():,.2f}"
)

print(
    "Net Transfer:",
    f"${claims_transactions_clean['NetTransferAmount'].sum():,.2f}"
)

Transfer Validation
Transfer In: $85,812,683.56
Transfer Out: $85,812,683.56
Net Transfer: $0.00


In [227]:
print("Charge vs Payment Validation")
print("=" * 40)

total_charges = claims_transactions_clean["ChargeAmount"].sum()
total_payments = claims_transactions_clean["PaymentAmount"].sum()

print(f"Total Charges:  ${total_charges:,.2f}")
print(f"Total Payments: ${total_payments:,.2f}")
print(
    f"Difference:     ${total_charges - total_payments:,.2f}"
)

Charge vs Payment Validation
Total Charges:  $437,111,182.35
Total Payments: $437,111,182.35
Difference:     $-0.00


In [228]:
claims_transactions_clean.to_csv(
    PROCESSED_DATA / "claims_transactions_clean.csv",
    index=False
)

print("claims_transactions_clean.csv exported successfully.")

claims_transactions_clean.csv exported successfully.


In [229]:
print("Claims Transaction Date Range")
print("=" * 40)

print(
    "Earliest:",
    claims_transactions_clean["FROMDATE"].min()
)

print(
    "Latest:",
    claims_transactions_clean["FROMDATE"].max()
)

Claims Transaction Date Range
Earliest: 1920-01-02T00:24:20Z
Latest: 2026-08-09T15:20:14Z


In [230]:
# ============================================================
# Claims Transactions - Convert Dates
# ============================================================

claims_transactions_clean["FROMDATE"] = pd.to_datetime(
    claims_transactions_clean["FROMDATE"],
    errors="coerce"
)

claims_transactions_clean["TODATE"] = pd.to_datetime(
    claims_transactions_clean["TODATE"],
    errors="coerce"
)

print("FROMDATE data type:")
print(claims_transactions_clean["FROMDATE"].dtype)

print("\nDate range:")
print("Earliest:", claims_transactions_clean["FROMDATE"].min())
print("Latest:", claims_transactions_clean["FROMDATE"].max())

FROMDATE data type:
datetime64[us, UTC]

Date range:
Earliest: 1920-01-02 00:24:20+00:00
Latest: 2026-08-09 15:20:14+00:00


In [231]:
# ============================================================
# Yearly Financial Profile
# ============================================================

yearly_financials = (
    claims_transactions_clean
    .groupby(claims_transactions_clean["FROMDATE"].dt.year)
    .agg(
        Transactions=("ID", "count"),
        Charges=("ChargeAmount", "sum"),
        Payments=("PaymentAmount", "sum"),
        Outstanding=("OutstandingAmount", "sum"),
        TransfersIn=("TransferInAmount", "sum"),
        TransfersOut=("TransferOutAmount", "sum")
    )
    .round(2)
)

yearly_financials

,Transactions,Charges,Payments,Outstanding,TransfersIn,TransfersOut
FROMDATE,,,,,,
1920,9,171.10,171.10,273.76,119.77,119.77
1921,18,735.65,735.65,273.60,136.80,136.80
1922,60,4695.29,4695.29,9390.58,4695.29,4695.29
1923,69,4539.14,4539.14,8429.43,4196.94,4196.94
1924,54,4754.00,4754.00,9332.58,4666.29,4666.29
...,...,...,...,...,...,...
2022,214885,34281360.13,34281360.13,19053865.95,7213871.61,7213871.61
2023,213506,33700919.28,33700919.28,18119693.65,6907044.56,6907044.56
2024,214976,33670564.63,33670564.63,18096512.73,6898855.51,6898855.51


In [232]:
yearly_financials[
    ["Transactions", "Charges", "Payments"]
].sort_values(
    "Charges",
    ascending=False
)

,Transactions,Charges,Payments
FROMDATE,,,
2021,236479,34694548.80,34694548.80
2025,221902,34410333.05,34410333.05
2022,214885,34281360.13,34281360.13
2023,213506,33700919.28,33700919.28
2024,214976,33670564.63,33670564.63
...,...,...,...
1921,18,735.65,735.65
1928,20,609.15,609.15
1933,16,444.70,444.70


In [233]:
yearly_financials.loc[[2020, 2021, 2022]]

,Transactions,Charges,Payments,Outstanding,TransfersIn,TransfersOut
FROMDATE,,,,,,
2020,209481,32185920.46,32185920.46,17915733.08,6880344.79,6880344.79
2021,236479,34694548.80,34694548.80,19820063.29,7563189.44,7563189.44
2022,214885,34281360.13,34281360.13,19053865.95,7213871.61,7213871.61


In [234]:
encounters["START"] = pd.to_datetime(
    encounters["START"],
    errors="coerce"
)

active_patients_2020_2022 = encounters[
    (encounters["START"] >= "2020-01-01") &
    (encounters["START"] < "2023-01-01")
]["PATIENT"].nunique()

print(
    "Active Patients 2020-2022:",
    f"{active_patients_2020_2022:,}"
)

Active Patients 2020-2022: 2,493


In [235]:
# ============================================================
# 2020-2022 Encounter Analysis
# ============================================================

encounters_analysis = encounters[
    (encounters["START"] >= "2020-01-01") &
    (encounters["START"] < "2023-01-01")
].copy()

print("2020-2022 Encounter Summary")
print("=" * 50)

print(f"Total Encounters: {len(encounters_analysis):,}")

print(
    "Unique Patients:",
    f"{encounters_analysis['PATIENT'].nunique():,}"
)

print(
    "Unique Organizations:",
    f"{encounters_analysis['ORGANIZATION'].nunique():,}"
)

print(
    "Unique Providers:",
    f"{encounters_analysis['PROVIDER'].nunique():,}"
)

2020-2022 Encounter Summary
Total Encounters: 37,289
Unique Patients: 2,493
Unique Organizations: 768
Unique Providers: 768


In [236]:
# Encounter volume by year

encounters_analysis["Year"] = (
    encounters_analysis["START"].dt.year
)

encounters_by_year = (
    encounters_analysis
    .groupby("Year")
    .agg(
        Encounters=("Id", "count"),
        Patients=("PATIENT", "nunique"),
        Organizations=("ORGANIZATION", "nunique"),
        Providers=("PROVIDER", "nunique")
    )
)

encounters_by_year

,Encounters,Patients,Organizations,Providers
Year,,,,
2020,11213,2259,662,662
2021,14444,2406,646,646
2022,11632,2309,675,675


In [237]:
# Encounter mix

encounters_analysis["ENCOUNTERCLASS"].value_counts()

ENCOUNTERCLASS
ambulatory    19409
outpatient     7341
wellness       6742
emergency      1493
urgentcare     1410
inpatient       366
home            202
virtual         144
snf             128
hospice          54
Name: count, dtype: int64

In [238]:
# ============================================================
# 2020-2022 Healthcare Utilization KPIs
# ============================================================

total_encounters = len(encounters_analysis)

active_patients = encounters_analysis["PATIENT"].nunique()

emergency_encounters = (
    encounters_analysis["ENCOUNTERCLASS"] == "emergency"
).sum()

inpatient_encounters = (
    encounters_analysis["ENCOUNTERCLASS"] == "inpatient"
).sum()

urgentcare_encounters = (
    encounters_analysis["ENCOUNTERCLASS"] == "urgentcare"
).sum()

virtual_encounters = (
    encounters_analysis["ENCOUNTERCLASS"] == "virtual"
).sum()

encounters_per_patient = (
    total_encounters / active_patients
)

emergency_rate = (
    emergency_encounters / total_encounters
)

inpatient_rate = (
    inpatient_encounters / total_encounters
)

print("Healthcare Utilization KPIs")
print("=" * 50)

print(f"Active Patients:       {active_patients:,}")
print(f"Total Encounters:      {total_encounters:,}")
print(f"Encounters / Patient:  {encounters_per_patient:.2f}")
print(f"Emergency Encounters:  {emergency_encounters:,}")
print(f"Emergency Rate:        {emergency_rate:.2%}")
print(f"Inpatient Encounters:  {inpatient_encounters:,}")
print(f"Inpatient Rate:        {inpatient_rate:.2%}")
print(f"Urgent Care:           {urgentcare_encounters:,}")
print(f"Virtual Encounters:    {virtual_encounters:,}")

Healthcare Utilization KPIs
Active Patients:       2,493
Total Encounters:      37,289
Encounters / Patient:  14.96
Emergency Encounters:  1,493
Emergency Rate:        4.00%
Inpatient Encounters:  366
Inpatient Rate:        0.98%
Urgent Care:           1,410
Virtual Encounters:    144


In [239]:
# ============================================================
# Length of Stay
# ============================================================

encounters_analysis["STOP"] = pd.to_datetime(
    encounters_analysis["STOP"],
    errors="coerce"
)

encounters_analysis["LengthOfStayDays"] = (
    (
        encounters_analysis["STOP"]
        - encounters_analysis["START"]
    ).dt.total_seconds()
    / 86400
)

print("Length of Stay Validation")
print("=" * 50)

print(
    "Negative LOS:",
    (encounters_analysis["LengthOfStayDays"] < 0).sum()
)

print(
    "Missing LOS:",
    encounters_analysis["LengthOfStayDays"].isna().sum()
)

print(
    "Average LOS:",
    f"{encounters_analysis['LengthOfStayDays'].mean():.2f} days"
)

print(
    "Median LOS:",
    f"{encounters_analysis['LengthOfStayDays'].median():.2f} days"
)

Length of Stay Validation
Negative LOS: 0
Missing LOS: 0
Average LOS: 0.22 days
Median LOS: 0.03 days


In [240]:
# ============================================================
# Inpatient Length of Stay
# ============================================================

inpatient = encounters_analysis[
    encounters_analysis["ENCOUNTERCLASS"] == "inpatient"
].copy()

print("Inpatient Length of Stay")
print("=" * 50)

print(f"Inpatient Encounters: {len(inpatient):,}")

print(
    f"Average Inpatient LOS: "
    f"{inpatient['LengthOfStayDays'].mean():.2f} days"
)

print(
    f"Median Inpatient LOS: "
    f"{inpatient['LengthOfStayDays'].median():.2f} days"
)

print(
    f"Maximum Inpatient LOS: "
    f"{inpatient['LengthOfStayDays'].max():.2f} days"
)

Inpatient Length of Stay
Inpatient Encounters: 366
Average Inpatient LOS: 5.12 days
Median Inpatient LOS: 3.78 days
Maximum Inpatient LOS: 24.84 days


In [241]:
inpatient["LengthOfStayDays"].describe()

count    366.000000
mean       5.115499
std        4.836932
min        1.000000
25%        1.000000
50%        3.784375
75%        7.000000
max       24.843553
Name: LengthOfStayDays, dtype: float64

In [242]:
# ============================================================
# 30-Day Readmission Analysis
# ============================================================

# Sort inpatient encounters by patient and admission date
inpatient = inpatient.sort_values(
    ["PATIENT", "START"]
).copy()

# Previous inpatient discharge for each patient
inpatient["PreviousDischarge"] = (
    inpatient
    .groupby("PATIENT")["STOP"]
    .shift(1)
)

# Days between previous discharge and current admission
inpatient["DaysSincePreviousDischarge"] = (
    inpatient["START"] - inpatient["PreviousDischarge"]
).dt.total_seconds() / 86400

# 30-day readmission flag
inpatient["Readmission30Days"] = (
    inpatient["DaysSincePreviousDischarge"].between(0, 30)
)

print("30-Day Readmission Analysis")
print("=" * 50)

print(
    "Inpatient Encounters:",
    f"{len(inpatient):,}"
)

print(
    "Readmissions within 30 days:",
    f"{inpatient['Readmission30Days'].sum():,}"
)

print(
    "Readmission Rate:",
    f"{inpatient['Readmission30Days'].mean():.2%}"
)

30-Day Readmission Analysis
Inpatient Encounters: 366
Readmissions within 30 days: 29
Readmission Rate: 7.92%


In [243]:
# ============================================================
# 2020-2022 Mortality Analysis
# ============================================================

patients_clean["DEATHDATE"] = pd.to_datetime(
    patients_clean["DEATHDATE"],
    errors="coerce"
)

deaths_2020_2022 = patients_clean[
    (patients_clean["DEATHDATE"] >= "2020-01-01") &
    (patients_clean["DEATHDATE"] < "2023-01-01")
].copy()

print("2020-2022 Mortality Analysis")
print("=" * 50)

print(
    "Deaths during 2020-2022:",
    f"{len(deaths_2020_2022):,}"
)

print(
    "Mortality Rate:",
    f"{len(deaths_2020_2022) / active_patients:.2%}"
)

2020-2022 Mortality Analysis
Deaths during 2020-2022: 43
Mortality Rate: 1.72%


In [244]:
# ============================================================
# 2020-2022 Mortality Analysis
# ============================================================

patients_clean["DEATHDATE"] = pd.to_datetime(
    patients_clean["DEATHDATE"],
    errors="coerce"
)

deaths_2020_2022 = patients_clean[
    (patients_clean["DEATHDATE"] >= "2020-01-01") &
    (patients_clean["DEATHDATE"] < "2023-01-01")
].copy()

print("2020-2022 Mortality Analysis")
print("=" * 50)

print(
    "Deaths during 2020-2022:",
    f"{len(deaths_2020_2022):,}"
)

print(
    "Mortality Rate:",
    f"{len(deaths_2020_2022) / active_patients:.2%}"
)

2020-2022 Mortality Analysis
Deaths during 2020-2022: 43
Mortality Rate: 1.72%


In [245]:
# Deaths by year

deaths_by_year = (
    deaths_2020_2022
    .assign(
        Year=deaths_2020_2022["DEATHDATE"].dt.year
    )
    .groupby("Year")
    .size()
    .rename("Deaths")
)

deaths_by_year

Year
2020    15
2021    15
2022    13
Name: Deaths, dtype: int64

In [246]:
# ============================================================
# 2020-2022 Claims Analysis
# ============================================================

claims["SERVICEDATE"] = pd.to_datetime(
    claims["SERVICEDATE"],
    errors="coerce"
)

claims_analysis = claims[
    (claims["SERVICEDATE"] >= "2020-01-01") &
    (claims["SERVICEDATE"] < "2023-01-01")
].copy()

print("Claims Analysis - 2020-2022")
print("=" * 50)

print(f"Total Claims: {len(claims_analysis):,}")

print(
    "Unique Patients:",
    f"{claims_analysis['PATIENTID'].nunique():,}"
)

print(
    "Unique Providers:",
    f"{claims_analysis['PROVIDERID'].nunique():,}"
)

Claims Analysis - 2020-2022
Total Claims: 67,567
Unique Patients: 2,493
Unique Providers: 768


In [247]:
# ============================================================
# Claims Financial Summary
# ============================================================

financial_summary = claims_analysis[
    [
        "OUTSTANDING1",
        "OUTSTANDING2",
        "OUTSTANDINGP"
    ]
].describe()

financial_summary

,OUTSTANDING1,OUTSTANDING2,OUTSTANDINGP
count,67567.000000,44723.000000,67567.000000
mean,0.241870,0.059553,0.060693
std,35.219864,3.502181,8.873956
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,9083.140000,211.380000,2270.790000


In [248]:
claims_analysis["TotalOutstanding"] = (
    claims_analysis["OUTSTANDING1"].fillna(0)
    + claims_analysis["OUTSTANDING2"].fillna(0)
    + claims_analysis["OUTSTANDINGP"].fillna(0)
)

print(
    "Total Outstanding:",
    f"${claims_analysis['TotalOutstanding'].sum():,.2f}"
)

Total Outstanding: $23,106.65


In [249]:
# ============================================================
# 2020-2022 Claims Transactions
# ============================================================

claims_transactions["FROMDATE"] = pd.to_datetime(
    claims_transactions["FROMDATE"],
    errors="coerce"
)

transactions_analysis = claims_transactions[
    (claims_transactions["FROMDATE"] >= "2020-01-01") &
    (claims_transactions["FROMDATE"] < "2023-01-01")
].copy()

print("Claims Transactions - 2020-2022")
print("=" * 50)

print(f"Total Transactions: {len(transactions_analysis):,}")

print(
    "Unique Claims:",
    f"{transactions_analysis['CLAIMID'].nunique():,}"
)

print(
    "Unique Patients:",
    f"{transactions_analysis['PATIENTID'].nunique():,}"
)

print("\nTransaction Types:")
print(
    transactions_analysis["TYPE"]
    .value_counts()
)

Claims Transactions - 2020-2022
Total Transactions: 660,845
Unique Claims: 67,566
Unique Patients: 2,493

Transaction Types:
TYPE
PAYMENT        246168
CHARGE         194363
TRANSFEROUT    110157
TRANSFERIN     110157
Name: count, dtype: int64


In [250]:
# ============================================================
# Financial Totals
# ============================================================

financial_totals = (
    transactions_analysis
    .groupby("TYPE")
    .agg(
        Transactions=("ID", "count"),
        Amount=("AMOUNT", "sum"),
        Payments=("PAYMENTS", "sum"),
        Adjustments=("ADJUSTMENTS", "sum"),
        Transfers=("TRANSFERS", "sum"),
        Outstanding=("OUTSTANDING", "sum")
    )
)

financial_totals

,Transactions,Amount,Payments,Adjustments,Transfers,Outstanding
TYPE,,,,,,
CHARGE,194363,1.011618e+08,0.000000e+00,0.0,0.00,0.00
PAYMENT,246168,0.000000e+00,1.011618e+08,0.0,0.00,13474850.64
TRANSFERIN,110157,2.165741e+07,0.000000e+00,0.0,21657405.84,21657405.84
TRANSFEROUT,110157,0.000000e+00,0.000000e+00,0.0,21657405.84,21657405.84


In [251]:
# ============================================================
# Executive Financial KPIs
# ============================================================

total_charges = transactions_analysis.loc[
    transactions_analysis["TYPE"] == "CHARGE",
    "AMOUNT"
].sum()

total_payments = transactions_analysis.loc[
    transactions_analysis["TYPE"] == "PAYMENT",
    "PAYMENTS"
].sum()

total_transfers = transactions_analysis[
    "TRANSFERS"
].sum()

total_outstanding = transactions_analysis[
    "OUTSTANDING"
].sum()

print("Healthcare Financial KPIs")
print("=" * 50)

print(f"Total Charges:       ${total_charges:,.2f}")
print(f"Total Payments:      ${total_payments:,.2f}")
print(f"Total Transfers:     ${total_transfers:,.2f}")
print(f"Total Outstanding:   ${total_outstanding:,.2f}")

Healthcare Financial KPIs
Total Charges:       $101,161,829.39
Total Payments:      $101,161,829.39
Total Transfers:     $43,314,811.68
Total Outstanding:   $56,789,662.32


In [252]:
# ============================================================
# Financial Performance Metrics
# ============================================================

payment_rate = (
    total_payments / total_charges
    if total_charges > 0 else 0
)

outstanding_rate = (
    total_outstanding / total_charges
    if total_charges > 0 else 0
)

avg_charge_per_encounter = (
    total_charges / total_encounters
)

avg_charge_per_patient = (
    total_charges / active_patients
)

print("Financial Performance Metrics")
print("=" * 50)

print(f"Payment Rate:             {payment_rate:.2%}")
print(f"Outstanding Rate:         {outstanding_rate:.2%}")
print(f"Average Charge/Encounter: ${avg_charge_per_encounter:,.2f}")
print(f"Average Charge/Patient:   ${avg_charge_per_patient:,.2f}")

Financial Performance Metrics
Payment Rate:             100.00%
Outstanding Rate:         56.14%
Average Charge/Encounter: $2,712.91
Average Charge/Patient:   $40,578.35


In [253]:
# ============================================================
# Financial Trends by Year
# ============================================================

transactions_analysis["Year"] = (
    transactions_analysis["FROMDATE"].dt.year
)

financial_by_year = (
    transactions_analysis
    .groupby(["Year", "TYPE"])
    .agg(
        Amount=("AMOUNT", "sum"),
        Payments=("PAYMENTS", "sum"),
        Transfers=("TRANSFERS", "sum"),
        Outstanding=("OUTSTANDING", "sum")
    )
    .reset_index()
)

financial_by_year

,Year,TYPE,Amount,Payments,Transfers,Outstanding
0,2020,CHARGE,32185920.46,0.00,0.00,0.00
1,2020,PAYMENT,0.00,32185920.46,0.00,4155043.50
2,2020,TRANSFERIN,6880344.79,0.00,6880344.79,6880344.79
3,2020,TRANSFEROUT,0.00,0.00,6880344.79,6880344.79
4,2021,CHARGE,34694548.80,0.00,0.00,0.00
5,2021,PAYMENT,0.00,34694548.80,0.00,4693684.41
6,2021,TRANSFERIN,7563189.44,0.00,7563189.44,7563189.44
7,2021,TRANSFEROUT,0.00,0.00,7563189.44,7563189.44
8,2022,CHARGE,34281360.13,0.00,0.00,0.00
9,2022,PAYMENT,0.00,34281360.13,0.00,4626122.73


In [254]:
conditions = pd.read_csv(
    RAW_DATA / "conditions.csv"
)

print(
    f"Conditions loaded: {len(conditions):,}"
)

conditions.head()

Conditions loaded: 102,851


,START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION
0,1991-10-31,NaN,0b879291-4e25-a926-1253-0604f2f8913f,0b879291-4e25-a926-4097-793c8006d6bc,http://snomed.info/sct,224299000,Received higher education (finding)
1,2013-09-28,NaN,ebde6fa6-7f57-0893-8c60-9bc114b60899,ebde6fa6-7f57-0893-748d-6eeeccfa2731,http://snomed.info/sct,40055000,Chronic sinusitis (disorder)
2,2018-10-01,2019-06-10,12021186-2496-a178-80b1-5c9e52adf9a3,12021186-2496-a178-c232-3afa8dcfd265,http://snomed.info/sct,314529007,Medication review due (situation)
3,2001-11-15,2017-09-21,0b879291-4e25-a926-1253-0604f2f8913f,0b879291-4e25-a926-3421-aadceefbf206,http://snomed.info/sct,160903007,Full-time employment (finding)
4,2010-09-13,NaN,229514f0-f87a-5e77-2a7b-c0112a2c5801,229514f0-f87a-5e77-d028-ea97e7a67de9,http://snomed.info/sct,224299000,Received higher education (finding)


In [255]:
conditions = pd.read_csv(
    RAW_DATA / "conditions.csv"
)

print(f"Conditions loaded: {len(conditions):,}")

conditions.head()

Conditions loaded: 102,851


,START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION
0,1991-10-31,NaN,0b879291-4e25-a926-1253-0604f2f8913f,0b879291-4e25-a926-4097-793c8006d6bc,http://snomed.info/sct,224299000,Received higher education (finding)
1,2013-09-28,NaN,ebde6fa6-7f57-0893-8c60-9bc114b60899,ebde6fa6-7f57-0893-748d-6eeeccfa2731,http://snomed.info/sct,40055000,Chronic sinusitis (disorder)
2,2018-10-01,2019-06-10,12021186-2496-a178-80b1-5c9e52adf9a3,12021186-2496-a178-c232-3afa8dcfd265,http://snomed.info/sct,314529007,Medication review due (situation)
3,2001-11-15,2017-09-21,0b879291-4e25-a926-1253-0604f2f8913f,0b879291-4e25-a926-3421-aadceefbf206,http://snomed.info/sct,160903007,Full-time employment (finding)
4,2010-09-13,NaN,229514f0-f87a-5e77-2a7b-c0112a2c5801,229514f0-f87a-5e77-d028-ea97e7a67de9,http://snomed.info/sct,224299000,Received higher education (finding)


In [256]:
conditions.info()

<class 'pandas.DataFrame'>
RangeIndex: 102851 entries, 0 to 102850
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   START        102851 non-null  str  
 1   STOP         76192 non-null   str  
 2   PATIENT      102851 non-null  str  
 3   ENCOUNTER    102851 non-null  str  
 4   SYSTEM       102851 non-null  str  
 5   CODE         102851 non-null  int64
 6   DESCRIPTION  102851 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.5 MB


In [257]:
conditions = pd.read_csv(
    RAW_DATA / "conditions.csv"
)

print(f"Conditions loaded: {len(conditions):,}")

conditions.head()

Conditions loaded: 102,851


,START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION
0,1991-10-31,NaN,0b879291-4e25-a926-1253-0604f2f8913f,0b879291-4e25-a926-4097-793c8006d6bc,http://snomed.info/sct,224299000,Received higher education (finding)
1,2013-09-28,NaN,ebde6fa6-7f57-0893-8c60-9bc114b60899,ebde6fa6-7f57-0893-748d-6eeeccfa2731,http://snomed.info/sct,40055000,Chronic sinusitis (disorder)
2,2018-10-01,2019-06-10,12021186-2496-a178-80b1-5c9e52adf9a3,12021186-2496-a178-c232-3afa8dcfd265,http://snomed.info/sct,314529007,Medication review due (situation)
3,2001-11-15,2017-09-21,0b879291-4e25-a926-1253-0604f2f8913f,0b879291-4e25-a926-3421-aadceefbf206,http://snomed.info/sct,160903007,Full-time employment (finding)
4,2010-09-13,NaN,229514f0-f87a-5e77-2a7b-c0112a2c5801,229514f0-f87a-5e77-d028-ea97e7a67de9,http://snomed.info/sct,224299000,Received higher education (finding)


In [258]:
conditions.info()

<class 'pandas.DataFrame'>
RangeIndex: 102851 entries, 0 to 102850
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   START        102851 non-null  str  
 1   STOP         76192 non-null   str  
 2   PATIENT      102851 non-null  str  
 3   ENCOUNTER    102851 non-null  str  
 4   SYSTEM       102851 non-null  str  
 5   CODE         102851 non-null  int64
 6   DESCRIPTION  102851 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.5 MB


In [259]:
# ============================================================
# Conditions Data Quality Profile
# ============================================================

print("Conditions Data Quality Profile")
print("=" * 50)

print(f"Rows: {len(conditions):,}")
print(f"Columns: {len(conditions.columns)}")
print(f"Duplicate Rows: {conditions.duplicated().sum():,}")

print("\nMissing Values:")
print(
    conditions.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nUnique Patients:")
print(f"{conditions['PATIENT'].nunique():,}")

print("\nUnique Conditions:")
print(f"{conditions['CODE'].nunique():,}")

print("\nUnique Descriptions:")
print(f"{conditions['DESCRIPTION'].nunique():,}")

Conditions Data Quality Profile
Rows: 102,851
Columns: 7
Duplicate Rows: 0

Missing Values:
STOP           26659
START              0
PATIENT            0
ENCOUNTER          0
SYSTEM             0
CODE               0
DESCRIPTION        0
dtype: int64

Unique Patients:
2,868

Unique Conditions:
283

Unique Descriptions:
283


In [260]:
# ============================================================
# Condition Frequency
# ============================================================

condition_frequency = (
    conditions["DESCRIPTION"]
    .value_counts()
)

condition_frequency.head(20)

DESCRIPTION
Medication review due (situation)                   20359
Stress (finding)                                     7837
Full-time employment (finding)                       7368
Gingivitis (disorder)                                7339
Part-time employment (finding)                       4487
Viral sinusitis (disorder)                           2974
Limited social contact (finding)                     2880
Social isolation (finding)                           2782
Not in labor force (finding)                         2608
Gingival disease (disorder)                          2201
Victim of intimate partner abuse (finding)           1893
Primary dental caries (disorder)                     1738
Acute viral pharyngitis (disorder)                   1684
Reports of violence in the environment (finding)     1539
Received higher education (finding)                  1359
Acute bronchitis (disorder)                          1350
Body mass index 30+ - obesity (finding)              1349
No

In [261]:
# ============================================================
# Conditions Cleaning
# ============================================================

conditions_clean = conditions.copy()

# Convert dates
conditions_clean["START"] = pd.to_datetime(
    conditions_clean["START"],
    errors="coerce"
)

conditions_clean["STOP"] = pd.to_datetime(
    conditions_clean["STOP"],
    errors="coerce"
)

# Create condition category from Synthea description
conditions_clean["ConditionCategory"] = (
    conditions_clean["DESCRIPTION"]
    .str.extract(r"\((.*?)\)$")[0]
    .str.lower()
    .str.strip()
)

print("Condition Categories")
print("=" * 50)

print(
    conditions_clean["ConditionCategory"]
    .value_counts(dropna=False)
)

Condition Categories
ConditionCategory
finding                    46001
disorder                   33213
situation                  22461
morphologic abnormality      492
panic) (finding              386
NaN                          165
person                       133
Name: count, dtype: int64


In [262]:
# Remove the Synthea classification from the description

conditions_clean["ConditionName"] = (
    conditions_clean["DESCRIPTION"]
    .str.replace(
        r"\s*\([^)]*\)$",
        "",
        regex=True
    )
    .str.strip()
)

conditions_clean[
    ["DESCRIPTION", "ConditionName", "ConditionCategory"]
].head(20)

,DESCRIPTION,ConditionName,ConditionCategory
0,Received higher education (finding),Received higher education,finding
1,Chronic sinusitis (disorder),Chronic sinusitis,disorder
2,Medication review due (situation),Medication review due,situation
3,Full-time employment (finding),Full-time employment,finding
4,Received higher education (finding),Received higher education,finding
5,Full-time employment (finding),Full-time employment,finding
6,Has a criminal record (finding),Has a criminal record,finding
7,Chronic sinusitis (disorder),Chronic sinusitis,disorder
8,Otitis media (disorder),Otitis media,disorder
9,Medication review due (situation),Medication review due,situation


In [263]:
# ============================================================
# 2020-2022 Conditions
# ============================================================

conditions_analysis = conditions_clean[
    (conditions_clean["START"] >= "2020-01-01") &
    (conditions_clean["START"] < "2023-01-01")
].copy()

print("Conditions Analysis - 2020-2022")
print("=" * 50)

print(
    f"Condition Records: {len(conditions_analysis):,}"
)

print(
    f"Unique Patients: "
    f"{conditions_analysis['PATIENT'].nunique():,}"
)

print(
    f"Unique Conditions: "
    f"{conditions_analysis['CODE'].nunique():,}"
)

Conditions Analysis - 2020-2022
Condition Records: 22,142
Unique Patients: 2,470
Unique Conditions: 240


In [264]:
conditions_analysis["ConditionCategory"].value_counts()

ConditionCategory
finding                    9274
disorder                   7445
situation                  5165
morphologic abnormality     125
panic) (finding              80
person                        7
Name: count, dtype: int64

In [265]:
# ============================================================
# Top Diagnoses by Unique Patients
# ============================================================

top_diagnoses = (
    conditions_analysis[
        conditions_analysis["ConditionCategory"] == "disorder"
    ]
    .groupby("ConditionName")["PATIENT"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)

top_diagnoses

ConditionName
Gingivitis                                                           1153
Viral sinusitis                                                       674
Gingival disease                                                      479
Acute viral pharyngitis                                               439
Primary dental caries                                                 416
Acute bronchitis                                                      342
Disease caused by severe acute respiratory syndrome coronavirus 2     230
Infection of tooth                                                    140
Laceration - injury                                                   122
Otitis media                                                          113
Fracture of bone                                                      110
Streptococcal sore throat                                             106
Anemia                                                                 83
Acute infective cystitis

In [266]:
ongoing_conditions = conditions_analysis[
    conditions_analysis["STOP"].isna()
]

print(
    "Ongoing Conditions:",
    f"{len(ongoing_conditions):,}"
)

print(
    "Patients with Ongoing Conditions:",
    f"{ongoing_conditions['PATIENT'].nunique():,}"
)

Ongoing Conditions: 2,386
Patients with Ongoing Conditions: 1,221


In [267]:
# Top 50 diagnoses
top_50_diagnoses = (
    conditions_analysis[
        conditions_analysis["ConditionCategory"] == "disorder"
    ]
    .groupby("ConditionName")["PATIENT"]
    .nunique()
    .sort_values(ascending=False)
    .head(50)
)

top_50_diagnoses

ConditionName
Gingivitis                                                           1153
Viral sinusitis                                                       674
Gingival disease                                                      479
Acute viral pharyngitis                                               439
Primary dental caries                                                 416
Acute bronchitis                                                      342
Disease caused by severe acute respiratory syndrome coronavirus 2     230
Infection of tooth                                                    140
Laceration - injury                                                   122
Otitis media                                                          113
Fracture of bone                                                      110
Streptococcal sore throat                                             106
Anemia                                                                 83
Acute infective cystitis

In [268]:
# ============================================================
# Clinical Condition Categorization
# ============================================================

def categorize_condition(condition):
    condition = condition.lower()

    # Dental
    if any(word in condition for word in [
        "gingivitis",
        "gingival",
        "dental",
        "tooth",
        "teeth",
        "caries"
    ]):
        return "Dental"

    # Respiratory
    elif any(word in condition for word in [
        "sinusitis",
        "sinus",
        "bronchitis",
        "pharyngitis",
        "pneumonia",
        "asthma",
        "respiratory",
        "otitis"
    ]):
        return "Respiratory"

    # Cardiovascular
    elif any(word in condition for word in [
        "heart",
        "cardiac",
        "hypertension",
        "ischemic",
        "coronary",
        "atrial",
        "cardiovascular"
    ]):
        return "Cardiovascular"

    # Metabolic / Endocrine
    elif any(word in condition for word in [
        "diabetes",
        "prediabetes",
        "obesity",
        "hyperlipidemia",
        "metabolic",
        "thyroid",
        "microalbuminuria"
    ]):
        return "Metabolic"

    # Musculoskeletal / Injury
    elif any(word in condition for word in [
        "fracture",
        "sprain",
        "injury",
        "laceration",
        "arthritis",
        "joint",
        "bone",
        "musculoskeletal",
        "concussion"
    ]):
        return "Musculoskeletal / Injury"

    # Genitourinary / Kidney
    elif any(word in condition for word in [
        "kidney",
        "renal",
        "cystitis",
        "urinary",
        "urinary tract",
        "bladder"
    ]):
        return "Genitourinary"

    # Neurological
    elif any(word in condition for word in [
        "neurolog",
        "seizure",
        "migraine",
        "epilepsy",
        "stroke",
        "brain",
        "dementia"
    ]):
        return "Neurological"

    # Women's Health
    elif any(word in condition for word in [
        "pregnancy",
        "miscarriage",
        "prenatal",
        "postpartum",
        "gynec",
        "menstrual"
    ]):
        return "Women's Health"

    # Infectious Disease
    elif any(word in condition for word in [
        "infection",
        "infective",
        "viral",
        "bacterial",
        "covid",
        "coronavirus",
        "streptococcal"
    ]):
        return "Infectious Disease"

    # Mental / Behavioral
    elif any(word in condition for word in [
        "stress",
        "anxiety",
        "depression",
        "mental",
        "behavior",
        "social isolation"
    ]):
        return "Mental / Behavioral"

    else:
        return "Other"


conditions_clean["ClinicalCategory"] = (
    conditions_clean["ConditionName"]
    .apply(categorize_condition)
)

print("Clinical Category Distribution")
print("=" * 50)

print(
    conditions_clean["ClinicalCategory"]
    .value_counts()
)

Clinical Category Distribution
ClinicalCategory
Other                       55198
Dental                      13538
Mental / Behavioral         11537
Respiratory                  8223
Metabolic                    4225
Musculoskeletal / Injury     3824
Women's Health               2274
Cardiovascular               1832
Genitourinary                1335
Neurological                  443
Infectious Disease            422
Name: count, dtype: int64


In [269]:
other_conditions = (
    conditions_clean[
        conditions_clean["ClinicalCategory"] == "Other"
    ]["ConditionName"]
    .value_counts()
)

print(f"Other conditions: {len(other_conditions)}")
print()
print(other_conditions.to_string())

Other conditions: 154

ConditionName
Medication review due                                         20359
Full-time employment                                           7368
Part-time employment                                           4487
Limited social contact                                         2880
Not in labor force                                             2608
Victim of intimate partner abuse                               1893
Reports of violence in the environment                         1539
Received higher education                                      1359
Unemployed                                                     1066
Anemia                                                         1026
Risk activity involvement                                       748
Chronic pain                                                    712
Educated to high school level                                   615
Has a criminal record                                           546
Chronic low

In [270]:
conditions_analysis = conditions_clean[
    (conditions_clean["START"] >= "2020-01-01") &
    (conditions_clean["START"] < "2023-01-01")
].copy()

print("2020-2022 Clinical Dataset")
print("=" * 50)

print(f"Records: {len(conditions_analysis):,}")
print(f"Patients: {conditions_analysis['PATIENT'].nunique():,}")
print(f"Conditions: {conditions_analysis['CODE'].nunique():,}")

2020-2022 Clinical Dataset
Records: 22,142
Patients: 2,470
Conditions: 240


In [271]:
patients_by_category = (
    conditions_analysis
    .groupby("ClinicalCategory")["PATIENT"]
    .nunique()
    .sort_values(ascending=False)
)

patients_by_category

ClinicalCategory
Other                       2367
Dental                      1446
Respiratory                 1398
Mental / Behavioral         1294
Musculoskeletal / Injury     501
Metabolic                    275
Women's Health               265
Genitourinary                160
Cardiovascular               114
Infectious Disease           106
Neurological                   8
Name: PATIENT, dtype: int64

In [272]:
top_diagnoses_by_category = (
    conditions_analysis
    .groupby(
        ["ClinicalCategory", "ConditionName"]
    )["PATIENT"]
    .nunique()
    .reset_index(name="UniquePatients")
    .sort_values(
        ["ClinicalCategory", "UniquePatients"],
        ascending=[True, False]
    )
)

top_diagnoses_by_category.head(30)

,ClinicalCategory,ConditionName,UniquePatients
0,Cardiovascular,Abnormal findings diagnostic imaging heart+cor...,59
7,Cardiovascular,Ischemic heart disease,59
3,Cardiovascular,Essential hypertension,52
5,Cardiovascular,History of coronary artery bypass grafting,23
2,Cardiovascular,Chronic congestive heart failure,3
4,Cardiovascular,Heart failure,2
1,Cardiovascular,Atrial fibrillation,1
6,Cardiovascular,Injury of heart,1
11,Dental,Gingivitis,1153
10,Dental,Gingival disease,479


In [273]:
# ============================================================
# Clinical Category Audit
# ============================================================

category_summary = (
    conditions_clean
    .groupby("ClinicalCategory")
    .agg(
        ConditionCount=("ConditionName", "nunique"),
        PatientCount=("PATIENT", "nunique"),
        RecordCount=("ConditionName", "size")
    )
    .sort_values("PatientCount", ascending=False)
)

category_summary

,ConditionCount,PatientCount,RecordCount
ClinicalCategory,,,
Other,154,2868,55198
Respiratory,16,2666,8223
Dental,10,2531,13538
Mental / Behavioral,6,2273,11537
Metabolic,14,1723,4225
Musculoskeletal / Injury,45,1516,3824
Women's Health,8,876,2274
Cardiovascular,8,875,1832
Genitourinary,14,615,1335


In [274]:
# ============================================================
# Review "Other"
# ============================================================

other_conditions = (
    conditions_clean[
        conditions_clean["ClinicalCategory"] == "Other"
    ][["ConditionName", "CODE"]]
    .drop_duplicates()
    .sort_values("ConditionName")
)

print(f"Other unique conditions: {len(other_conditions):,}")

other_conditions.to_string(index=False)

Other unique conditions: 154


"                                             ConditionName            CODE\n                             Acquired coagulation disorder       234466008\n                       Acquired immune deficiency syndrome        62479008\n          Acute ST segment elevation myocardial infarction       401303003\n                                   Acute allergic reaction       241929008\n                              Acute deep venous thrombosis 132281000119108\n                                    Acute myeloid leukemia        91861009\n      Acute non-ST segment elevation myocardial infarction       401314000\n                                  Acute pulmonary embolism       706870000\n                           Adolescent idiopathic scoliosis       203646004\n                                                Alcoholism         7200002\n                                         Alveolitis of jaw        61804006\n                                       Alzheimer's disease        26929004\n           

In [275]:
cardio_conditions = (
    conditions_clean[
        conditions_clean["ClinicalCategory"] == "Cardiovascular"
    ][["ConditionName", "CODE"]]
    .drop_duplicates()
    .sort_values("ConditionName")
)

print(f"Cardiovascular unique conditions: {len(cardio_conditions):,}")

cardio_conditions.to_string(index=False)

Cardiovascular unique conditions: 8


'                                               ConditionName      CODE\nAbnormal findings diagnostic imaging heart+coronary circulat 274531002\n                                         Atrial fibrillation  49436004\n                            Chronic congestive heart failure  88805009\n                                      Essential hypertension  59621000\n                                               Heart failure  84114007\n                  History of coronary artery bypass grafting 399261000\n                                             Injury of heart  86175003\n                                      Ischemic heart disease 414545008'

In [276]:
print(cardio_conditions.to_string(index=False))

                                               ConditionName      CODE
Abnormal findings diagnostic imaging heart+coronary circulat 274531002
                                         Atrial fibrillation  49436004
                            Chronic congestive heart failure  88805009
                                      Essential hypertension  59621000
                                               Heart failure  84114007
                  History of coronary artery bypass grafting 399261000
                                             Injury of heart  86175003
                                      Ischemic heart disease 414545008


In [277]:
print(other_conditions.to_string(index=False))

                                             ConditionName            CODE
                             Acquired coagulation disorder       234466008
                       Acquired immune deficiency syndrome        62479008
          Acute ST segment elevation myocardial infarction       401303003
                                   Acute allergic reaction       241929008
                              Acute deep venous thrombosis 132281000119108
                                    Acute myeloid leukemia        91861009
      Acute non-ST segment elevation myocardial infarction       401314000
                                  Acute pulmonary embolism       706870000
                           Adolescent idiopathic scoliosis       203646004
                                                Alcoholism         7200002
                                         Alveolitis of jaw        61804006
                                       Alzheimer's disease        26929004
                         

In [278]:
print(other_conditions.head(75).to_string(index=False))

                                       ConditionName            CODE
                       Acquired coagulation disorder       234466008
                 Acquired immune deficiency syndrome        62479008
    Acute ST segment elevation myocardial infarction       401303003
                             Acute allergic reaction       241929008
                        Acute deep venous thrombosis 132281000119108
                              Acute myeloid leukemia        91861009
Acute non-ST segment elevation myocardial infarction       401314000
                            Acute pulmonary embolism       706870000
                     Adolescent idiopathic scoliosis       203646004
                                          Alcoholism         7200002
                                   Alveolitis of jaw        61804006
                                 Alzheimer's disease        26929004
                                              Anemia       271737000
                          Aortic v

In [279]:
print(other_conditions.iloc[75:150].to_string(index=False))

                                             ConditionName           CODE
                                 Major depressive disorder      370143000
                              Malignant neoplasm of breast      254837009
                               Malignant neoplasm of colon      363406005
                                     Medication review due      314529007
                    Metastatic malignant neoplasm to colon       94260004
                 Metastatic malignant neoplasm to prostate       94503003
                                             Misuses drugs      361055000
                                Mitral valve regurgitation       48724000
                                     Mitral valve stenosis       79619009
                                          Multiple myeloma      109989006
                                               Muscle pain       68962001
                                     Myocardial infarction       22298006
                                      

In [280]:
print(other_conditions.iloc[150:].to_string(index=False))

                   ConditionName      CODE
                      Unemployed  73438004
Victim of intimate partner abuse 706893006
                Vomiting symptom 249497008
                        Wheezing  56018004


In [281]:
print(other_conditions.to_string(index=False))

                                             ConditionName            CODE
                             Acquired coagulation disorder       234466008
                       Acquired immune deficiency syndrome        62479008
          Acute ST segment elevation myocardial infarction       401303003
                                   Acute allergic reaction       241929008
                              Acute deep venous thrombosis 132281000119108
                                    Acute myeloid leukemia        91861009
      Acute non-ST segment elevation myocardial infarction       401314000
                                  Acute pulmonary embolism       706870000
                           Adolescent idiopathic scoliosis       203646004
                                                Alcoholism         7200002
                                         Alveolitis of jaw        61804006
                                       Alzheimer's disease        26929004
                         

In [282]:
print(other_conditions.to_string(index=False))

                                             ConditionName            CODE
                             Acquired coagulation disorder       234466008
                       Acquired immune deficiency syndrome        62479008
          Acute ST segment elevation myocardial infarction       401303003
                                   Acute allergic reaction       241929008
                              Acute deep venous thrombosis 132281000119108
                                    Acute myeloid leukemia        91861009
      Acute non-ST segment elevation myocardial infarction       401314000
                                  Acute pulmonary embolism       706870000
                           Adolescent idiopathic scoliosis       203646004
                                                Alcoholism         7200002
                                         Alveolitis of jaw        61804006
                                       Alzheimer's disease        26929004
                         

In [283]:
def categorize_condition(condition):
    condition = condition.lower()

    # Dental
    if any(word in condition for word in [
        "gingivitis",
        "gingival",
        "dental",
        "tooth",
        "teeth",
        "caries"
    ]):
        return "Dental"

    # Cardiovascular
    elif any(word in condition for word in [
        "myocardial infarction",
        "heart failure",
        "heart disease",
        "ischemic heart",
        "hypertension",
        "coronary",
        "atrial fibrillation",
        "aortic valve",
        "pulmonary embolism",
        "deep vein thrombosis",
        "cerebrovascular",
        "stroke",
        "cardiac",
        "cardiovascular"
    ]):
        return "Cardiovascular"

    # Respiratory
    elif any(word in condition for word in [
        "sinusitis",
        "sinus",
        "bronchitis",
        "pharyngitis",
        "pneumonia",
        "asthma",
        "respiratory",
        "otitis",
        "wheezing"
    ]):
        return "Respiratory"

    # Metabolic / Endocrine
    elif any(word in condition for word in [
        "diabetes",
        "prediabetes",
        "obesity",
        "body mass index",
        "hyperlipidemia",
        "metabolic",
        "thyroid",
        "microalbuminuria"
    ]):
        return "Metabolic / Endocrine"

    # Musculoskeletal / Injury
    elif any(word in condition for word in [
        "fracture",
        "sprain",
        "injury",
        "laceration",
        "arthritis",
        "scoliosis",
        "joint",
        "bone",
        "musculoskeletal",
        "concussion",
        "bullet wound"
    ]):
        return "Musculoskeletal / Injury"

    # Genitourinary
    elif any(word in condition for word in [
        "kidney",
        "renal",
        "cystitis",
        "urinary",
        "bladder",
        "renal transplant"
    ]):
        return "Genitourinary"

    # Neurological
    elif any(word in condition for word in [
        "neurolog",
        "seizure",
        "migraine",
        "epilepsy",
        "alzheimer",
        "cerebral palsy",
        "brain",
        "dementia"
    ]):
        return "Neurological"

    # Women's Health
    elif any(word in condition for word in [
        "pregnancy",
        "miscarriage",
        "prenatal",
        "postpartum",
        "gynec",
        "menstrual",
        "blighted ovum"
    ]):
        return "Women's Health"

    # Infectious Disease
    elif any(word in condition for word in [
        "infection",
        "infective",
        "viral",
        "bacterial",
        "covid",
        "coronavirus",
        "streptococcal",
        "acquired immune deficiency",
        "aids"
    ]):
        return "Infectious Disease"

    # Mental / Behavioral
    elif any(word in condition for word in [
        "stress",
        "anxiety",
        "depression",
        "mental",
        "behavior",
        "alcoholism"
    ]):
        return "Mental / Behavioral"

    # Oncology
    elif any(word in condition for word in [
        "cancer",
        "carcinoma",
        "leukemia",
        "lymphoma",
        "melanoma",
        "tumor",
        "neoplasm"
    ]):
        return "Oncology"

    # Hematologic
    elif any(word in condition for word in [
        "anemia",
        "hemoglobin",
        "blood disorder"
    ]):
        return "Hematologic"

    # Gastrointestinal
    elif any(word in condition for word in [
        "appendicitis",
        "vomiting",
        "nausea",
        "gastro",
        "intestinal",
        "bowel",
        "stomach",
        "bleeding from anus"
    ]):
        return "Gastrointestinal"

    # Dermatological
    elif any(word in condition for word in [
        "dermatitis",
        "eczema",
        "skin",
        "rash",
        "allergic reaction"
    ]):
        return "Dermatological"

    # Symptoms
    elif any(word in condition for word in [
        "symptom",
        "wheezing",
        "vomiting",
        "pain",
        "fever",
        "cough"
    ]):
        return "Symptoms"

    # Social Determinants of Health
    elif any(word in condition for word in [
        "unemployed",
        "employment",
        "education",
        "educated to",
        "social isolation",
        "intimate partner abuse",
        "victim of intimate partner abuse",
        "not in labor force",
        "homeless",
        "transport problem"
    ]):
        return "Social / SDOH"

    # Hematologic
    elif any(word in condition for word in [
        "coagulation disorder",
        "neutropenia",
        "anemia",
        "blood disorder"
    ]):
        return "Hematologic"

    # Care / End-of-Life
    elif any(word in condition for word in [
        "died in hospice",
        "hospice"
    ]):
        return "Care / End-of-Life"

    else:
        return "Other"

In [284]:
conditions_clean["ClinicalCategory"] = (
    conditions_clean["ConditionName"]
    .apply(categorize_condition)
)

print(
    conditions_clean["ClinicalCategory"]
    .value_counts()
)

ClinicalCategory
Other                       30834
Social / SDOH               22831
Dental                      13538
Mental / Behavioral          8769
Respiratory                  8268
Metabolic / Endocrine        4244
Musculoskeletal / Injury     3913
Women's Health               2326
Cardiovascular               2270
Symptoms                     1935
Genitourinary                1335
Hematologic                  1030
Neurological                  528
Infectious Disease            423
Oncology                      354
Dermatological                165
Gastrointestinal               74
Care / End-of-Life             14
Name: count, dtype: int64


In [285]:
other_conditions = (
    conditions_clean[
        conditions_clean["ClinicalCategory"] == "Other"
    ][["ConditionName", "CODE"]]
    .drop_duplicates()
    .sort_values("ConditionName")
)

print(f"Remaining Other conditions: {len(other_conditions):,}")
print(other_conditions.to_string(index=False))

Remaining Other conditions: 86
                                      ConditionName            CODE
                       Acute deep venous thrombosis 132281000119108
                                  Alveolitis of jaw        61804006
                   Child attention deficit disorder       192127007
                                              Chill        43724002
                                Chronic hepatitis C       128302006
     Chronic paralysis due to lesion of spinal cord       698754002
                         Congenital uterine anomaly        37849005
                               Dependent drug abuse         6525002
                               Dribbling from mouth        62718007
                                            Dyspnea       267036007
                                           Dystonia        15802004
                               Excessive salivation        53827007
                                            Fatigue        84229001
                 

In [286]:
# ============================================================
# Clinical Category Overrides
# Specific mappings for conditions not captured by keywords
# ============================================================

category_overrides = {

    # Cardiovascular
    "Acute deep venous thrombosis": "Cardiovascular",
    "Tricuspid valve regurgitation": "Cardiovascular",

    # Dental
    "Alveolitis of jaw": "Dental",
    "Torus palatinus": "Dental",

    # Mental / Behavioral
    "Child attention deficit disorder": "Mental / Behavioral",
    "Dependent drug abuse": "Mental / Behavioral",

    # Infectious Disease
    "Chronic hepatitis C": "Infectious Disease",
    "Toxoplasma gondii antibody detected": "Infectious Disease",
    "History of tuberculosis": "Infectious Disease",

    # Neurological
    "Chronic paralysis due to lesion of spinal cord": "Neurological",
    "Dystonia": "Neurological",

    # Women's Health
    "Congenital uterine anomaly": "Women's Health",
    "Fetus with chromosomal abnormality": "Women's Health",
    "History of tubal ligation": "Women's Health",

    # Musculoskeletal / Injury
    "Fibromyalgia": "Musculoskeletal / Injury",
    "Full thickness burn": "Musculoskeletal / Injury",
    "Gout": "Musculoskeletal / Injury",
    "Gunshot wound": "Musculoskeletal / Injury",

    # Symptoms
    "Chill": "Symptoms",
    "Dribbling from mouth": "Symptoms",
    "Dyspnea": "Symptoms",
    "Excessive salivation": "Symptoms",
    "Fatigue": "Symptoms",
    "Headache": "Symptoms",
    "Unable to swallow saliva": "Symptoms",

    # Gastrointestinal
    "History of appendectomy": "Gastrointestinal",

    # Hematologic
    "History of peripheral stem cell transplant": "Hematologic",

    # Genitourinary
    "History of renal transplant": "Genitourinary",

    # Social / SDOH
    "Has a criminal record": "Social / SDOH",
    "Transport problem": "Social / SDOH"
}

conditions_clean["ClinicalCategory"] = (
    conditions_clean["ConditionName"]
    .map(category_overrides)
    .fillna(conditions_clean["ClinicalCategory"])
)

print("Category overrides applied:", len(category_overrides))

Category overrides applied: 30


In [287]:
other_conditions = (
    conditions_clean[
        conditions_clean["ClinicalCategory"] == "Other"
    ][["ConditionName", "CODE"]]
    .drop_duplicates()
    .sort_values("ConditionName")
)

print(
    f"Remaining Other conditions: {len(other_conditions):,}"
)

print(
    other_conditions.to_string(index=False)
)

Remaining Other conditions: 59
                                      ConditionName       CODE
                             Housing unsatisfactory  105531004
                                      Hyperglycemia   80394007
                               Hypertriglyceridemia  302870006
                                          Hypoxemia  389087006
                                    Impacted molars  196416002
                           Infectious mediastinitis  312157006
                            Intellectual disability  110359009
                   Lack of access to transportation  713458007
                             Limited social contact  423315002
                                      Loss of taste   36955009
                          Major depressive disorder  370143000
                              Medication review due  314529007
                                      Misuses drugs  361055000
                         Mitral valve regurgitation   48724000
                        

In [289]:
conditions_clean["ClinicalCategory"] = (
    conditions_clean["ConditionName"]
    .map(category_overrides)
    .fillna(conditions_clean["ClinicalCategory"])
)

In [290]:
other_conditions = (
    conditions_clean[
        conditions_clean["ClinicalCategory"] == "Other"
    ][["ConditionName", "CODE"]]
    .drop_duplicates()
    .sort_values("ConditionName")
)

print(
    f"Remaining Other conditions: {len(other_conditions):,}"
)

print(other_conditions.to_string(index=False))

Remaining Other conditions: 59
                                      ConditionName       CODE
                             Housing unsatisfactory  105531004
                                      Hyperglycemia   80394007
                               Hypertriglyceridemia  302870006
                                          Hypoxemia  389087006
                                    Impacted molars  196416002
                           Infectious mediastinitis  312157006
                            Intellectual disability  110359009
                   Lack of access to transportation  713458007
                             Limited social contact  423315002
                                      Loss of taste   36955009
                          Major depressive disorder  370143000
                              Medication review due  314529007
                                      Misuses drugs  361055000
                         Mitral valve regurgitation   48724000
                        

In [291]:
conditions_clean["ClinicalCategory"] = (
    conditions_clean["ClinicalCategory"]
    .replace("Other", "Unclassified")
)

In [292]:
conditions_clean["ClinicalCategory"].value_counts()

ClinicalCategory
Unclassified                29119
Social / SDOH               23377
Dental                      13693
Mental / Behavioral          8948
Respiratory                  8268
Metabolic / Endocrine        4244
Musculoskeletal / Injury     4039
Women's Health               2644
Cardiovascular               2287
Symptoms                     2137
Genitourinary                1335
Hematologic                  1036
Neurological                  534
Infectious Disease            428
Oncology                      354
Gastrointestinal              229
Dermatological                165
Care / End-of-Life             14
Name: count, dtype: int64

In [293]:
unclassified = (
    conditions_clean[
        conditions_clean["ClinicalCategory"] == "Unclassified"
    ]
)

print(f"Unclassified records: {len(unclassified):,}")
print(
    f"Unclassified patients: "
    f"{unclassified['PATIENT'].nunique():,}"
)

Unclassified records: 29,119
Unclassified patients: 2,868


In [294]:
unclassified_summary = (
    unclassified
    .groupby("ConditionName")["PATIENT"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="UniquePatients")
)

unclassified_summary

,ConditionName,UniquePatients
0,Medication review due,2868
1,Limited social contact,1564
2,Reports of violence in the environment,970
3,Risk activity involvement,701
4,Housing unsatisfactory,336
5,Lack of access to transportation,283
6,Misuses drugs,237
7,Hypertriglyceridemia,212
8,Polyp of colon,185
9,Serving in military service,154


In [295]:
# Rename "Other" to "Unclassified"
conditions_clean["ClinicalCategory"] = (
    conditions_clean["ClinicalCategory"]
    .replace("Other", "Unclassified")
)

print("Categories:")
print(conditions_clean["ClinicalCategory"].value_counts())

Categories:
ClinicalCategory
Unclassified                29119
Social / SDOH               23377
Dental                      13693
Mental / Behavioral          8948
Respiratory                  8268
Metabolic / Endocrine        4244
Musculoskeletal / Injury     4039
Women's Health               2644
Cardiovascular               2287
Symptoms                     2137
Genitourinary                1335
Hematologic                  1036
Neurological                  534
Infectious Disease            428
Oncology                      354
Gastrointestinal              229
Dermatological                165
Care / End-of-Life             14
Name: count, dtype: int64


In [296]:
# Unclassified records and patients

unclassified = conditions_clean[
    conditions_clean["ClinicalCategory"] == "Unclassified"
]

print("Unclassified Validation")
print("=" * 40)

print(
    "Unclassified records:",
    f"{len(unclassified):,}"
)

print(
    "Unclassified patients:",
    f"{unclassified['PATIENT'].nunique():,}"
)

Unclassified Validation
Unclassified records: 29,119
Unclassified patients: 2,868


In [297]:
# Most common Unclassified conditions by unique patients

unclassified_summary = (
    unclassified
    .groupby("ConditionName")["PATIENT"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="UniquePatients")
)

print(
    unclassified_summary.to_string(index=False)
)

                                      ConditionName  UniquePatients
                              Medication review due            2868
                             Limited social contact            1564
             Reports of violence in the environment             970
                          Risk activity involvement             701
                             Housing unsatisfactory             336
                   Lack of access to transportation             283
                                      Misuses drugs             237
                               Hypertriglyceridemia             212
                                     Polyp of colon             185
                        Serving in military service             154
                                       Osteoporosis             141
                                            Refugee             133
                                    Impacted molars             133
                                     Sleep disor

In [298]:
# ============================================================
# Final Clinical Category Overrides
# ============================================================

final_overrides = {

    # Care / Administrative
    "Medication review due": "Care / Administrative",

    # Social Determinants of Health
    "Limited social contact": "Social / SDOH",
    "Reports of violence in the environment": "Social / SDOH",
    "Risk activity involvement": "Social / SDOH",
    "Housing unsatisfactory": "Social / SDOH",
    "Lack of access to transportation": "Social / SDOH",
    "Serving in military service": "Social / SDOH",
    "Refugee": "Social / SDOH",

    # Mental / Behavioral
    "Misuses drugs": "Mental / Behavioral",

    # Metabolic / Endocrine
    "Hypertriglyceridemia": "Metabolic / Endocrine",
    "Hyperglycemia": "Metabolic / Endocrine",

    # Gastrointestinal
    "Polyp of colon": "Gastrointestinal",
    "Recurrent rectal polyp": "Gastrointestinal",

    # Musculoskeletal / Injury
    "Osteoporosis": "Musculoskeletal / Injury",

    # Dental
    "Impacted molars": "Dental",

    # Respiratory
    "Sleep disorder": "Respiratory",
    "Obstructive sleep apnea syndrome": "Respiratory",
    "Perennial allergic rhinitis": "Respiratory",

    # Symptoms
    "Overdose": "Symptoms",
    "Loss of taste": "Symptoms",
    "Sputum finding": "Symptoms",

    # Infectious Disease
    "Sepsis": "Infectious Disease",

    # Women's Health
    "Sterilization requested": "Women's Health",
    "Pre-eclampsia": "Women's Health",

    # Cardiovascular
    "Mitral valve stenosis": "Cardiovascular",
    "Pulmonic valve regurgitation": "Cardiovascular",

    # Genitourinary
    "Retention of urine": "Genitourinary",

    # Neurological
    "Spina bifida occulta": "Neurological"
}

# Apply the overrides
conditions_clean["ClinicalCategory"] = (
    conditions_clean["ConditionName"]
    .map(final_overrides)
    .fillna(conditions_clean["ClinicalCategory"])
)

print(
    f"Overrides applied: {len(final_overrides)}"
)

Overrides applied: 28


In [299]:
# Verify final overrides

override_check = (
    conditions_clean[
        conditions_clean["ConditionName"].isin(final_overrides.keys())
    ][
        ["ConditionName", "ClinicalCategory"]
    ]
    .drop_duplicates()
    .sort_values("ConditionName")
)

print(override_check.to_string(index=False))

                         ConditionName         ClinicalCategory
                Housing unsatisfactory            Social / SDOH
                         Hyperglycemia    Metabolic / Endocrine
                  Hypertriglyceridemia    Metabolic / Endocrine
                       Impacted molars                   Dental
      Lack of access to transportation            Social / SDOH
                Limited social contact            Social / SDOH
                         Loss of taste                 Symptoms
                 Medication review due    Care / Administrative
                         Misuses drugs      Mental / Behavioral
                 Mitral valve stenosis           Cardiovascular
      Obstructive sleep apnea syndrome              Respiratory
                          Osteoporosis Musculoskeletal / Injury
                              Overdose                 Symptoms
           Perennial allergic rhinitis              Respiratory
                        Polyp of colon  

In [300]:
# Recalculate Unclassified conditions

unclassified = conditions_clean[
    conditions_clean["ClinicalCategory"] == "Unclassified"
]

print("Unclassified Validation")
print("=" * 40)

print(
    "Unclassified records:",
    f"{len(unclassified):,}"
)

print(
    "Unclassified patients:",
    f"{unclassified['PATIENT'].nunique():,}"
)

Unclassified Validation
Unclassified records: 546
Unclassified patients: 470


In [301]:
# Most common remaining Unclassified conditions

unclassified_summary = (
    unclassified
    .groupby("ConditionName")["PATIENT"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="UniquePatients")
)

print(
    unclassified_summary.to_string(index=False)
)

                                      ConditionName  UniquePatients
                                Pulmonary emphysema              43
                                       Opioid abuse              41
Perennial allergic rhinitis with seasonal variation              40
                                        Sleep apnea              37
                             Partial thickness burn              36
                               Smokes tobacco daily              35
                                          Hypoxemia              33
                                        Sore throat              29
                         Seasonal allergic rhinitis              25
                          Major depressive disorder              22
                         Rupture of patellar tendon              22
                           Tear of meniscus of knee              20
                                       Septic shock              16
                             Sepsis caused by vi

In [302]:
# ============================================================
# Remaining Unclassified Conditions
# ============================================================

unclassified_summary = (
    conditions_clean[
        conditions_clean["ClinicalCategory"] == "Unclassified"
    ]
    .groupby("ConditionName")["PATIENT"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="UniquePatients")
)

print(
    f"Remaining Unclassified conditions: "
    f"{len(unclassified_summary):,}"
)

print(
    unclassified_summary.to_string(index=False)
)

Remaining Unclassified conditions: 31
                                      ConditionName  UniquePatients
                                Pulmonary emphysema              43
                                       Opioid abuse              41
Perennial allergic rhinitis with seasonal variation              40
                                        Sleep apnea              37
                             Partial thickness burn              36
                               Smokes tobacco daily              35
                                          Hypoxemia              33
                                        Sore throat              29
                         Seasonal allergic rhinitis              25
                          Major depressive disorder              22
                         Rupture of patellar tendon              22
                           Tear of meniscus of knee              20
                                       Septic shock              16
          

In [303]:
# ============================================================
# Final Unclassified Condition Mappings
# ============================================================

final_condition_overrides = {

    # Respiratory
    "Pulmonary emphysema": "Respiratory",
    "Perennial allergic rhinitis with seasonal variation": "Respiratory",
    "Sleep apnea": "Respiratory",
    "Hypoxemia": "Respiratory",
    "Sore throat": "Respiratory",
    "Seasonal allergic rhinitis": "Respiratory",

    # Mental / Behavioral
    "Opioid abuse": "Mental / Behavioral",
    "Smokes tobacco daily": "Mental / Behavioral",
    "Major depressive disorder": "Mental / Behavioral",

    # Musculoskeletal / Injury
    "Partial thickness burn": "Musculoskeletal / Injury",
    "Rupture of patellar tendon": "Musculoskeletal / Injury",
    "Tear of meniscus of knee": "Musculoskeletal / Injury",
    "Primary fibromyalgia syndrome": "Musculoskeletal / Injury",

    # Infectious Disease
    "Septic shock": "Infectious Disease",
    "Sepsis caused by virus": "Infectious Disease",
    "Infectious mediastinitis": "Infectious Disease",

    # Gastrointestinal
    "Protracted diarrhea": "Gastrointestinal",
    "Rupture of appendix": "Gastrointestinal",

    # Cardiovascular
    "Mitral valve regurgitation": "Cardiovascular",
    "Pulmonic valve stenosis": "Cardiovascular",

    # Dental
    "Tongue tie": "Dental",
    "Torus mandibularis": "Dental",

    # Neurological
    "Spasticity": "Neurological",
    "Intellectual disability": "Neurological",
    "Poor muscle tone": "Neurological",

    # Symptoms
    "Shock": "Symptoms",

    # Social / SDOH
    "Social migrant": "Social / SDOH"
}

# Apply final mappings
conditions_clean["ClinicalCategory"] = (
    conditions_clean["ConditionName"]
    .map(final_condition_overrides)
    .fillna(conditions_clean["ClinicalCategory"])
)

print(
    f"Final mappings applied: "
    f"{len(final_condition_overrides)}"
)

Final mappings applied: 27


In [304]:
# ============================================================
# Final Unclassified Audit
# ============================================================

unclassified = conditions_clean[
    conditions_clean["ClinicalCategory"] == "Unclassified"
]

unclassified_summary = (
    unclassified
    .groupby("ConditionName")["PATIENT"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="UniquePatients")
)

print(
    f"Remaining Unclassified conditions: "
    f"{len(unclassified_summary):,}"
)

print(
    f"Unclassified records: "
    f"{len(unclassified):,}"
)

print(
    f"Unclassified patients: "
    f"{unclassified['PATIENT'].nunique():,}"
)

print("\nRemaining conditions:")
print(
    unclassified_summary.to_string(index=False)
)

Remaining Unclassified conditions: 4
Unclassified records: 24
Unclassified patients: 24

Remaining conditions:
         ConditionName  UniquePatients
      Multiple myeloma               6
      Nasal congestion               6
Preinfarction syndrome               6
        Pyelonephritis               6


In [305]:
# ============================================================
# Final 4 Clinical Category Mappings
# ============================================================

last_overrides = {
    "Multiple myeloma": "Oncology",
    "Nasal congestion": "Respiratory",
    "Preinfarction syndrome": "Cardiovascular",
    "Pyelonephritis": "Genitourinary"
}

conditions_clean["ClinicalCategory"] = (
    conditions_clean["ConditionName"]
    .map(last_overrides)
    .fillna(conditions_clean["ClinicalCategory"])
)

print("Final mappings applied:", len(last_overrides))

Final mappings applied: 4


In [306]:
# ============================================================
# FINAL UNCLASSIFIED VALIDATION
# ============================================================

unclassified = conditions_clean[
    conditions_clean["ClinicalCategory"] == "Unclassified"
]

print("FINAL UNCLASSIFIED VALIDATION")
print("=" * 50)

print(
    "Unclassified records:",
    f"{len(unclassified):,}"
)

print(
    "Unclassified patients:",
    f"{unclassified['PATIENT'].nunique():,}"
)

print(
    "Unclassified conditions:",
    f"{unclassified['ConditionName'].nunique():,}"
)

FINAL UNCLASSIFIED VALIDATION
Unclassified records: 0
Unclassified patients: 0
Unclassified conditions: 0


In [307]:
conditions_clean["ClinicalCategory"].value_counts()

ClinicalCategory
Social / SDOH               29454
Care / Administrative       20359
Dental                      13840
Mental / Behavioral          9297
Respiratory                  8752
Metabolic / Endocrine        4562
Musculoskeletal / Injury     4268
Women's Health               2780
Symptoms                     2708
Cardiovascular               2313
Genitourinary                1343
Hematologic                  1036
Infectious Disease            558
Neurological                  557
Gastrointestinal              485
Oncology                      360
Dermatological                165
Care / End-of-Life             14
Name: count, dtype: int64

In [308]:
conditions_clean["ClinicalCategory"].value_counts()

ClinicalCategory
Social / SDOH               29454
Care / Administrative       20359
Dental                      13840
Mental / Behavioral          9297
Respiratory                  8752
Metabolic / Endocrine        4562
Musculoskeletal / Injury     4268
Women's Health               2780
Symptoms                     2708
Cardiovascular               2313
Genitourinary                1343
Hematologic                  1036
Infectious Disease            558
Neurological                  557
Gastrointestinal              485
Oncology                      360
Dermatological                165
Care / End-of-Life             14
Name: count, dtype: int64

In [309]:
print("Rows:", len(conditions_clean))
print("Patients:", conditions_clean["PATIENT"].nunique())
print("Duplicate rows:", conditions_clean.duplicated().sum())
print(
    "Unclassified:",
    (conditions_clean["ClinicalCategory"] == "Unclassified").sum()
)

Rows: 102851
Patients: 2868
Duplicate rows: 0
Unclassified: 0


In [310]:
conditions_clean.to_csv(
    PROCESSED_DATA / "conditions_clean.csv",
    index=False
)

print("conditions_clean.csv exported successfully.")

conditions_clean.csv exported successfully.


In [311]:
print(
    "File exists:",
    (PROCESSED_DATA / "conditions_clean.csv").exists()
)

File exists: True


In [312]:
print("FINAL CONDITIONS VALIDATION")
print("=" * 50)

print("Rows:", f"{len(conditions_clean):,}")
print("Unique patients:", f"{conditions_clean['PATIENT'].nunique():,}")
print("Duplicate rows:", conditions_clean.duplicated().sum())
print("Missing PATIENT:", conditions_clean["PATIENT"].isna().sum())
print("Missing ConditionName:", conditions_clean["ConditionName"].isna().sum())
print(
    "Unclassified:",
    (conditions_clean["ClinicalCategory"] == "Unclassified").sum()
)

FINAL CONDITIONS VALIDATION
Rows: 102,851
Unique patients: 2,868
Duplicate rows: 0
Missing PATIENT: 0
Missing ConditionName: 0
Unclassified: 0


In [314]:
# ============================================================
# Load Procedures Dataset
# ============================================================

procedures = pd.read_csv(
    RAW_DATA / "procedures.csv"
)

print(f"Procedures loaded: {len(procedures):,}")
procedures.head()

Procedures loaded: 460,245


,START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION,BASE_COST,REASONCODE,REASONDESCRIPTION
0,2016-12-28T22:10:35Z,2016-12-28T22:25:35Z,ebde6fa6-7f57-0893-8c60-9bc114b60899,ebde6fa6-7f57-0893-71d1-af1294188ab6,http://snomed.info/sct,430193006,Medication reconciliation (procedure),215.7,NaN,NaN
1,2016-12-28T22:10:35Z,2016-12-28T22:22:28Z,ebde6fa6-7f57-0893-8c60-9bc114b60899,ebde6fa6-7f57-0893-71d1-af1294188ab6,http://snomed.info/sct,103697008,Patient referral for dental care (procedure),431.4,NaN,NaN
2,2017-09-21T14:16:01Z,2017-09-21T15:15:36Z,0b879291-4e25-a926-1253-0604f2f8913f,0b879291-4e25-a926-a3ce-f26441bd6410,http://snomed.info/sct,710824005,Assessment of health and social care needs (pr...,431.4,NaN,NaN
3,2017-09-25T06:07:54Z,2017-09-25T06:47:54Z,229514f0-f87a-5e77-2a7b-c0112a2c5801,229514f0-f87a-5e77-1b28-c8afae480b6c,http://snomed.info/sct,710824005,Assessment of health and social care needs (pr...,431.4,NaN,NaN
4,2017-09-25T06:47:54Z,2017-09-25T07:13:00Z,229514f0-f87a-5e77-2a7b-c0112a2c5801,229514f0-f87a-5e77-1b28-c8afae480b6c,http://snomed.info/sct,710841007,Assessment of anxiety (procedure),431.4,NaN,NaN


In [315]:
# ============================================================
# Procedures Data Quality Profile
# ============================================================

print("Procedures Data Quality Profile")
print("=" * 50)

print(f"Rows: {len(procedures):,}")
print(f"Columns: {len(procedures.columns)}")
print(f"Duplicate Rows: {procedures.duplicated().sum():,}")

print("\nColumn Information")
print("=" * 50)

print(procedures.dtypes)

print("\nMissing Values")
print("=" * 50)

missing = (
    procedures.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_pct = (
    procedures.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_profile = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct.round(2),
    "Data Type": procedures.dtypes.astype(str)
})

missing_profile

Procedures Data Quality Profile
Rows: 460,245
Columns: 10
Duplicate Rows: 0

Column Information
START                    str
STOP                     str
PATIENT                  str
ENCOUNTER                str
SYSTEM                   str
CODE                   int64
DESCRIPTION              str
BASE_COST            float64
REASONCODE           float64
REASONDESCRIPTION        str
dtype: object

Missing Values


,Missing Count,Missing %,Data Type
BASE_COST,0,0.00,float64
CODE,0,0.00,int64
DESCRIPTION,0,0.00,str
ENCOUNTER,0,0.00,str
PATIENT,0,0.00,str
REASONCODE,245732,53.39,float64
REASONDESCRIPTION,245732,53.39,str
START,0,0.00,str
STOP,0,0.00,str
SYSTEM,0,0.00,str


In [316]:
print("Procedures Uniqueness")
print("=" * 50)

print(
    "Unique Patients:",
    f"{procedures['PATIENT'].nunique():,}"
)

print(
    "Unique Encounters:",
    f"{procedures['ENCOUNTER'].nunique():,}"
)

print(
    "Unique Procedure Codes:",
    f"{procedures['CODE'].nunique():,}"
)

print(
    "Unique Procedure Descriptions:",
    f"{procedures['DESCRIPTION'].nunique():,}"
)

Procedures Uniqueness
Unique Patients: 2,856
Unique Encounters: 119,963
Unique Procedure Codes: 368
Unique Procedure Descriptions: 368


In [317]:
procedures["START"] = pd.to_datetime(
    procedures["START"],
    errors="coerce"
)

procedures["STOP"] = pd.to_datetime(
    procedures["STOP"],
    errors="coerce"
)

print("Procedures Date Range")
print("=" * 50)

print("Earliest:", procedures["START"].min())
print("Latest:", procedures["START"].max())

print("Missing START:", procedures["START"].isna().sum())
print("Missing STOP:", procedures["STOP"].isna().sum())

Procedures Date Range
Earliest: 1922-04-29 02:42:35+00:00
Latest: 2026-08-09 15:20:14+00:00
Missing START: 0
Missing STOP: 0


In [318]:
print("Top Procedures")
print("=" * 50)

top_procedures = (
    procedures["DESCRIPTION"]
    .value_counts()
    .head(20)
)

print(top_procedures)

Top Procedures
DESCRIPTION
Depression screening (procedure)                                                                   45872
Renal dialysis (procedure)                                                                         29299
Assessment of health and social care needs (procedure)                                             27723
Medication reconciliation (procedure)                                                              19451
Assessment of substance use (procedure)                                                            18551
Patient referral for dental care (procedure)                                                       16369
Dental consultation and report (procedure)                                                         15746
Oral health education (procedure)                                                                  15263
Dental care (regime/therapy)                                                                       15162
Removal of supragingival pla

In [319]:
print("Procedure Cost Validation")
print("=" * 50)

print("Negative costs:", (procedures["BASE_COST"] < 0).sum())
print("Missing costs:", procedures["BASE_COST"].isna().sum())

print("\nCost Summary")
print(procedures["BASE_COST"].describe())

Procedure Cost Validation
Negative costs: 0
Missing costs: 0

Cost Summary
count    460245.000000
mean        980.688527
std        2247.558519
min           0.320000
25%         431.400000
50%         431.400000
75%         431.400000
max      161615.000000
Name: BASE_COST, dtype: float64


In [320]:
procedures["ProcedureDurationHours"] = (
    procedures["STOP"] - procedures["START"]
).dt.total_seconds() / 3600

print("Procedure Duration Validation")
print("=" * 50)

print(
    "Negative durations:",
    (procedures["ProcedureDurationHours"] < 0).sum()
)

print(
    "Zero-duration procedures:",
    (procedures["ProcedureDurationHours"] == 0).sum()
)

print(
    "Missing duration:",
    procedures["ProcedureDurationHours"].isna().sum()
)

print("\nDuration Summary")
print(
    procedures["ProcedureDurationHours"].describe()
)

Procedure Duration Validation
Negative durations: 1
Zero-duration procedures: 48
Missing duration: 0

Duration Summary
count    460245.000000
mean          0.651745
std           5.260589
min          -0.228611
25%           0.250000
50%           0.374167
75%           0.586667
max        1386.000000
Name: ProcedureDurationHours, dtype: float64


In [321]:
procedures_analysis = procedures[
    (procedures["START"] >= "2020-01-01") &
    (procedures["START"] < "2023-01-01")
].copy()

print("Procedures 2020-2022")
print("=" * 50)

print("Records:", f"{len(procedures_analysis):,}")
print(
    "Unique patients:",
    f"{procedures_analysis['PATIENT'].nunique():,}"
)
print(
    "Unique procedures:",
    f"{procedures_analysis['CODE'].nunique():,}"
)
print(
    "Total procedure cost:",
    f"${procedures_analysis['BASE_COST'].sum():,.2f}"
)

Procedures 2020-2022
Records: 112,988
Unique patients: 2,482
Unique procedures: 322
Total procedure cost: $105,077,497.14


In [322]:
# ============================================================
# Investigate Negative Procedure Duration
# ============================================================

negative_duration = procedures[
    procedures["ProcedureDurationHours"] < 0
].copy()

negative_duration[
    [
        "START",
        "STOP",
        "PATIENT",
        "ENCOUNTER",
        "CODE",
        "DESCRIPTION",
        "BASE_COST",
        "ProcedureDurationHours"
    ]
]

,START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,BASE_COST,ProcedureDurationHours
422965,2024-07-31 01:16:14+00:00,2024-07-31 01:02:31+00:00,b1dba107-b34c-8d6f-599d-c5691910b4f8,b1dba107-b34c-8d6f-49f9-8aa9246e78b0,410011004,Administration of anesthesia AND/OR sedation (...,431.4,-0.228611


In [323]:
# ============================================================
# Longest Procedures
# ============================================================

procedures.nlargest(
    10,
    "ProcedureDurationHours"
)[
    [
        "START",
        "STOP",
        "PATIENT",
        "ENCOUNTER",
        "CODE",
        "DESCRIPTION",
        "ProcedureDurationHours"
    ]
]

,START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,ProcedureDurationHours
349730,2024-12-14 06:39:19+00:00,2025-02-10 00:39:19+00:00,d652cf26-3ad6-72bc-b801-38b779d817bb,d652cf26-3ad6-72bc-7131-1fa03b7afbf1,308481009,Referral to orthopedic surgeon (procedure),1386.0
297780,2021-06-11 16:51:22+00:00,2021-08-07 06:51:22+00:00,d37365d3-a9b0-866b-87ab-1731feb5c1e6,d37365d3-a9b0-866b-402e-1cfed8aafd53,308481009,Referral to orthopedic surgeon (procedure),1358.0
188499,2020-01-24 21:16:17+00:00,2020-03-05 17:16:17+00:00,9933d858-61da-6b01-1cd8-5d0fe6cc05db,9933d858-61da-6b01-01e0-031f43418c19,308481009,Referral to orthopedic surgeon (procedure),980.0
233103,2018-01-26 13:27:02+00:00,2018-03-08 04:27:02+00:00,a7e74ea9-800d-58c3-0367-be714404e415,a7e74ea9-800d-58c3-7aac-e5058dac718e,308481009,Referral to orthopedic surgeon (procedure),975.0
94219,2021-05-05 22:20:55+00:00,2021-06-10 23:20:55+00:00,20672ebb-bd90-778c-ff41-d6da595aec79,20672ebb-bd90-778c-8db1-441218dfb270,308481009,Referral to orthopedic surgeon (procedure),865.0
26209,2019-07-09 08:07:25+00:00,2019-08-13 14:07:25+00:00,3d0395d5-56b0-1d41-e530-01b5769dc162,3d0395d5-56b0-1d41-6d5a-ca887d2ae7e9,308481009,Referral to orthopedic surgeon (procedure),846.0
61763,2019-08-14 11:33:10+00:00,2019-09-17 13:33:10+00:00,e844b9b6-c7e8-22db-2a18-307c0d660b70,e844b9b6-c7e8-22db-9053-977a8d3ecea5,308481009,Referral to orthopedic surgeon (procedure),818.0
42043,2023-10-09 15:34:04+00:00,2023-11-08 18:34:04+00:00,2be6a5b6-c8c4-7d0f-36a8-750b0aaf3358,2be6a5b6-c8c4-7d0f-ad93-3bcb21c8aa05,308481009,Referral to orthopedic surgeon (procedure),723.0
333163,2015-05-25 11:01:02+00:00,2015-06-17 01:01:02+00:00,7c91be2d-154d-e256-cf9c-02167b2b2c0c,7c91be2d-154d-e256-d2dd-a368d7fdde03,133899007,Postoperative care (regime/therapy),542.0
197138,2020-01-29 10:52:19+00:00,2020-02-19 09:52:19+00:00,1fd2b779-58a9-aa5f-ac2b-a4d2f90925ff,1fd2b779-58a9-aa5f-9c9e-b93d235db4ee,133899007,Postoperative care (regime/therapy),503.0


In [324]:
negative_duration = procedures[
    procedures["ProcedureDurationHours"] < 0
].copy()

negative_duration[
    [
        "START",
        "STOP",
        "PATIENT",
        "ENCOUNTER",
        "CODE",
        "DESCRIPTION",
        "BASE_COST",
        "ProcedureDurationHours"
    ]
]

,START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,BASE_COST,ProcedureDurationHours
422965,2024-07-31 01:16:14+00:00,2024-07-31 01:02:31+00:00,b1dba107-b34c-8d6f-599d-c5691910b4f8,b1dba107-b34c-8d6f-49f9-8aa9246e78b0,410011004,Administration of anesthesia AND/OR sedation (...,431.4,-0.228611


In [325]:
# ============================================================
# Create Clean Procedures Dataset
# ============================================================

procedures_clean = procedures[
    procedures["ProcedureDurationHours"] >= 0
].copy()

print("Procedures Cleaning")
print("=" * 50)

print(f"Original records: {len(procedures):,}")
print(f"Clean records: {len(procedures_clean):,}")
print(
    f"Records removed: "
    f"{len(procedures) - len(procedures_clean):,}"
)

Procedures Cleaning
Original records: 460,245
Clean records: 460,244
Records removed: 1


In [326]:
print("Procedures Clean Validation")
print("=" * 50)

print(
    "Negative durations:",
    (procedures_clean["ProcedureDurationHours"] < 0).sum()
)

print(
    "Duplicate rows:",
    procedures_clean.duplicated().sum()
)

print(
    "Missing patients:",
    procedures_clean["PATIENT"].isna().sum()
)

print(
    "Missing descriptions:",
    procedures_clean["DESCRIPTION"].isna().sum()
)

print(
    "Missing costs:",
    procedures_clean["BASE_COST"].isna().sum()
)

Procedures Clean Validation
Negative durations: 0
Duplicate rows: 0
Missing patients: 0
Missing descriptions: 0
Missing costs: 0


In [327]:
procedures_analysis = procedures_clean[
    (procedures_clean["START"] >= "2020-01-01") &
    (procedures_clean["START"] < "2023-01-01")
].copy()

print("Procedures 2020-2022")
print("=" * 50)

print(f"Records: {len(procedures_analysis):,}")
print(
    f"Unique patients: "
    f"{procedures_analysis['PATIENT'].nunique():,}"
)
print(
    f"Unique procedures: "
    f"{procedures_analysis['CODE'].nunique():,}"
)
print(
    f"Total procedure cost: "
    f"${procedures_analysis['BASE_COST'].sum():,.2f}"
)

Procedures 2020-2022
Records: 112,988
Unique patients: 2,482
Unique procedures: 322
Total procedure cost: $105,077,497.14


In [328]:
# ============================================================
# Procedures Analysis Dataset: 2020-2022
# ============================================================

procedures_analysis = procedures_clean[
    (procedures_clean["START"] >= "2020-01-01") &
    (procedures_clean["START"] < "2023-01-01")
].copy()

print("Procedures 2020-2022")
print("=" * 50)

print(f"Records: {len(procedures_analysis):,}")
print(
    f"Unique patients: "
    f"{procedures_analysis['PATIENT'].nunique():,}"
)
print(
    f"Unique encounters: "
    f"{procedures_analysis['ENCOUNTER'].nunique():,}"
)
print(
    f"Unique procedures: "
    f"{procedures_analysis['CODE'].nunique():,}"
)
print(
    f"Total procedure cost: "
    f"${procedures_analysis['BASE_COST'].sum():,.2f}"
)

Procedures 2020-2022
Records: 112,988
Unique patients: 2,482
Unique encounters: 27,899
Unique procedures: 322
Total procedure cost: $105,077,497.14


In [329]:
print("Top Procedures: 2020-2022")
print("=" * 50)

procedures_analysis["DESCRIPTION"].value_counts().head(20)

Top Procedures: 2020-2022


DESCRIPTION
Depression screening (procedure)                                                                   11388
Assessment of health and social care needs (procedure)                                              6755
Medication reconciliation (procedure)                                                               4804
Renal dialysis (procedure)                                                                          4690
Assessment of substance use (procedure)                                                             4672
Patient referral for dental care (procedure)                                                        4090
Dental consultation and report (procedure)                                                          3989
Oral health education (procedure)                                                                   3863
Dental care (regime/therapy)                                                                        3831
Removal of supragingival plaque and calculu

In [330]:
procedures_by_year = (
    procedures_analysis
    .assign(Year=procedures_analysis["START"].dt.year)
    .groupby("Year")
    .agg(
        Procedures=("CODE", "count"),
        UniquePatients=("PATIENT", "nunique"),
        TotalCost=("BASE_COST", "sum")
    )
    .reset_index()
)

procedures_by_year

,Year,Procedures,UniquePatients,TotalCost
0,2020,36978,2121,33772653.20
1,2021,37640,2105,35382411.44
2,2022,38370,2159,35922432.50


In [331]:
print("Procedure Cost Metrics: 2020-2022")
print("=" * 50)

print(
    f"Total Cost: "
    f"${procedures_analysis['BASE_COST'].sum():,.2f}"
)

print(
    f"Average Cost: "
    f"${procedures_analysis['BASE_COST'].mean():,.2f}"
)

print(
    f"Median Cost: "
    f"${procedures_analysis['BASE_COST'].median():,.2f}"
)

print(
    f"Maximum Cost: "
    f"${procedures_analysis['BASE_COST'].max():,.2f}"
)

Procedure Cost Metrics: 2020-2022
Total Cost: $105,077,497.14
Average Cost: $929.99
Median Cost: $431.40
Maximum Cost: $108,025.04


In [332]:
print("FINAL PROCEDURES VALIDATION")
print("=" * 55)

print(f"Rows: {len(procedures_clean):,}")
print(f"Unique patients: {procedures_clean['PATIENT'].nunique():,}")
print(f"Duplicate rows: {procedures_clean.duplicated().sum():,}")
print(
    "Negative durations:",
    (procedures_clean["ProcedureDurationHours"] < 0).sum()
)
print(
    "Missing PATIENT:",
    procedures_clean["PATIENT"].isna().sum()
)
print(
    "Missing DESCRIPTION:",
    procedures_clean["DESCRIPTION"].isna().sum()
)
print(
    "Missing BASE_COST:",
    procedures_clean["BASE_COST"].isna().sum()
)

FINAL PROCEDURES VALIDATION
Rows: 460,244
Unique patients: 2,856
Duplicate rows: 0
Negative durations: 0
Missing PATIENT: 0
Missing DESCRIPTION: 0
Missing BASE_COST: 0


In [333]:
# ============================================================
# Export Clean Procedures Dataset
# ============================================================

procedures_clean.to_csv(
    PROCESSED_DATA / "procedures_clean.csv",
    index=False
)

print("procedures_clean.csv exported successfully.")

procedures_clean.csv exported successfully.


In [334]:
print(
    "File exists:",
    (PROCESSED_DATA / "procedures_clean.csv").exists()
)

File exists: True


In [335]:
# ============================================================
# Load Providers Dataset
# ============================================================

providers = pd.read_csv(
    RAW_DATA / "providers.csv"
)

print(f"Providers loaded: {len(providers):,}")
providers.head()

Providers loaded: 1,000


,Id,ORGANIZATION,NAME,GENDER,SPECIALITY,ADDRESS,CITY,STATE,ZIP,LAT,LON,ENCOUNTERS,PROCEDURES
0,a476b13c-d009-39d9-910b-967f85e134a9,74ab949d-17ac-3309-83a0-13b4405c66aa,Gabriel934 Reilly981,F,GENERAL PRACTICE,881 Main Street,Fitchburg,MA,1420,42.586487,-71.805210,6177,0
1,b1b4bdb9-3ca0-35bf-8085-c7ce229d8fef,da92d3fc-5445-3825-a937-043ef0d6ecd0,Neal874 Cruickshank494,M,GENERAL PRACTICE,336 GRATTAN ST,CHICOPEE,MA,10201314,42.166298,-72.590941,14,0
2,584c54fc-84fd-3e8f-b407-bb2b5f1bc6d6,9d7fb5a1-bf03-3928-a8b3-a2fceba4bd2c,Del587 Hoeger474,M,GENERAL PRACTICE,188 MAIN ST,WILMINGTON,MA,18872046,42.558685,-71.181566,20,0
3,0d4cdc25-2779-3e52-96cf-5da7be52f147,588f6ce6-b8db-3588-8189-29db2680a313,Tiffaney302 Brakus656,F,GENERAL PRACTICE,461 WALNUT AVE,JAMAICA PLAIN,MA,21302331,42.311588,-71.098001,423,0
4,6772eb7d-0000-39cf-9db8-a17d61fb301c,324b4137-57a0-3ae0-89db-1c33f57ae0c1,Marquis364 Satterfield305,M,GENERAL PRACTICE,134 NORTH ST,NORTH READING,MA,18641315,42.589234,-71.105465,2,0


In [336]:
providers.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Id            1000 non-null   str    
 1   ORGANIZATION  1000 non-null   str    
 2   NAME          1000 non-null   str    
 3   GENDER        1000 non-null   str    
 4   SPECIALITY    1000 non-null   str    
 5   ADDRESS       1000 non-null   str    
 6   CITY          1000 non-null   str    
 7   STATE         1000 non-null   str    
 8   ZIP           1000 non-null   int64  
 9   LAT           1000 non-null   float64
 10  LON           1000 non-null   float64
 11  ENCOUNTERS    1000 non-null   int64  
 12  PROCEDURES    1000 non-null   int64  
dtypes: float64(2), int64(3), str(8)
memory usage: 101.7 KB


In [337]:
print("Providers Data Quality Profile")
print("=" * 50)

print(f"Rows: {len(providers):,}")
print(f"Columns: {len(providers.columns)}")
print(f"Duplicate Rows: {providers.duplicated().sum():,}")

print("\nMissing Values")
print("=" * 50)

missing = providers.isna().sum().sort_values(ascending=False)

missing_pct = (
    providers.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct.round(2),
    "Data Type": providers.dtypes.astype(str)
})

Providers Data Quality Profile
Rows: 1,000
Columns: 13
Duplicate Rows: 0

Missing Values


,Missing Count,Missing %,Data Type
Id,0,0.0,str
ORGANIZATION,0,0.0,str
NAME,0,0.0,str
GENDER,0,0.0,str
SPECIALITY,0,0.0,str
ADDRESS,0,0.0,str
CITY,0,0.0,str
STATE,0,0.0,str
ZIP,0,0.0,int64
LAT,0,0.0,float64


In [338]:
print("Providers Validation")
print("=" * 50)

print("Unique Provider IDs:", providers["Id"].nunique())
print("Unique Provider Names:", providers["NAME"].nunique())
print("Unique Organizations:", providers["ORGANIZATION"].nunique())
print("Unique Specialties:", providers["SPECIALITY"].nunique())

print("\nProvider Gender Distribution")
print("=" * 50)

print(providers["GENDER"].value_counts())

print("\nTop Specialties")
print("=" * 50)

print(providers["SPECIALITY"].value_counts().head(15))

Providers Validation
Unique Provider IDs: 1000
Unique Provider Names: 1000
Unique Organizations: 1000
Unique Specialties: 1

Provider Gender Distribution
GENDER
F    521
M    479
Name: count, dtype: int64

Top Specialties
SPECIALITY
GENERAL PRACTICE    1000
Name: count, dtype: int64


In [339]:
print("Provider Geographic Validation")
print("=" * 50)

print(
    "Invalid latitude:",
    ((providers["LAT"] < -90) | (providers["LAT"] > 90)).sum()
)

print(
    "Invalid longitude:",
    ((providers["LON"] < -180) | (providers["LON"] > 180)).sum()
)

print(
    "Missing coordinates:",
    providers[["LAT", "LON"]].isna().any(axis=1).sum()
)

Provider Geographic Validation
Invalid latitude: 0
Invalid longitude: 0
Missing coordinates: 0


In [340]:
print("Provider Activity")
print("=" * 50)

print(
    "Total Encounters:",
    f"{providers['ENCOUNTERS'].sum():,}"
)

print(
    "Total Procedures:",
    f"{providers['PROCEDURES'].sum():,}"
)

print("\nMost Active Providers")
print("=" * 50)

providers[
    ["NAME", "SPECIALITY", "ENCOUNTERS", "PROCEDURES"]
].sort_values(
    "ENCOUNTERS",
    ascending=False
).head(15)

Provider Activity
Total Encounters: 472,595
Total Procedures: 0

Most Active Providers


,NAME,SPECIALITY,ENCOUNTERS,PROCEDURES
628,José Emilio366 Botello137,GENERAL PRACTICE,9615,0
592,Carlton317 Koch169,GENERAL PRACTICE,8227,0
170,Sharron285 Batz141,GENERAL PRACTICE,7895,0
848,Madelaine318 Walker122,GENERAL PRACTICE,7783,0
121,Ariane992 Pagac496,GENERAL PRACTICE,6795,0
231,Cleveland582 Kuphal363,GENERAL PRACTICE,6757,0
0,Gabriel934 Reilly981,GENERAL PRACTICE,6177,0
186,Johna806 Klein929,GENERAL PRACTICE,5961,0
884,Leonarda398 Schumm995,GENERAL PRACTICE,5858,0
911,Daniel959 Wolff180,GENERAL PRACTICE,5727,0


In [341]:
print("Provider ID Validation")
print("=" * 50)

print("Total providers:", len(providers))
print("Unique Provider IDs:", providers["Id"].nunique())
print("Duplicate Provider IDs:", providers["Id"].duplicated().sum())

Provider ID Validation
Total providers: 1000
Unique Provider IDs: 1000
Duplicate Provider IDs: 0


In [342]:
print("Provider Geographic Validation")
print("=" * 50)

print(
    "Invalid latitude:",
    ((providers["LAT"] < -90) | (providers["LAT"] > 90)).sum()
)

print(
    "Invalid longitude:",
    ((providers["LON"] < -180) | (providers["LON"] > 180)).sum()
)

print(
    "Missing coordinates:",
    providers[["LAT", "LON"]].isna().any(axis=1).sum()
)

Provider Geographic Validation
Invalid latitude: 0
Invalid longitude: 0
Missing coordinates: 0


In [343]:
# ============================================================
# Create Clean Providers Dataset
# ============================================================

providers_clean = providers.copy()

print("FINAL PROVIDERS VALIDATION")
print("=" * 50)

print(f"Rows: {len(providers_clean):,}")
print(f"Columns: {len(providers_clean.columns)}")
print(f"Unique Provider IDs: {providers_clean['Id'].nunique():,}")
print(f"Duplicate rows: {providers_clean.duplicated().sum():,}")
print(f"Missing values: {providers_clean.isna().sum().sum():,}")
print(
    "Invalid coordinates:",
    (
        (providers_clean["LAT"] < -90) |
        (providers_clean["LAT"] > 90) |
        (providers_clean["LON"] < -180) |
        (providers_clean["LON"] > 180)
    ).sum()
)

FINAL PROVIDERS VALIDATION
Rows: 1,000
Columns: 13
Unique Provider IDs: 1,000
Duplicate rows: 0
Missing values: 0
Invalid coordinates: 0


In [344]:
# ============================================================
# Export Clean Providers Dataset
# ============================================================

providers_clean.to_csv(
    PROCESSED_DATA / "providers_clean.csv",
    index=False
)

print("providers_clean.csv exported successfully.")

print(
    "File exists:",
    (PROCESSED_DATA / "providers_clean.csv").exists()
)

providers_clean.csv exported successfully.
File exists: True


In [345]:
# ============================================================
# Load Organizations Dataset
# ============================================================

organizations = pd.read_csv(
    RAW_DATA / "organizations.csv"
)

print(f"Organizations loaded: {len(organizations):,}")
organizations.head()

Organizations loaded: 1,000


,Id,NAME,ADDRESS,CITY,STATE,ZIP,LAT,LON,PHONE,REVENUE,UTILIZATION
0,74ab949d-17ac-3309-83a0-13b4405c66aa,Fitchburg Outpatient Clinic,881 Main Street,Fitchburg,MA,1420,42.586487,-71.805210,978-342-9781 Or 978-342-9781,0.0,6177
1,da92d3fc-5445-3825-a937-043ef0d6ecd0,TRINITY HOME CARE INC.,336 GRATTAN ST,CHICOPEE,MA,10201314,42.166298,-72.590941,4033313294,0.0,14
2,9d7fb5a1-bf03-3928-a8b3-a2fceba4bd2c,CARING HEARTS HOMECARE INC,188 MAIN ST,WILMINGTON,MA,18872046,42.558685,-71.181566,9786585104,0.0,20
3,588f6ce6-b8db-3588-8189-29db2680a313,BOSTON HEALTH CARE FOR THE HOMELESS PROGRAM INC,461 WALNUT AVE,JAMAICA PLAIN,MA,21302331,42.311588,-71.098001,8576541550,0.0,423
4,324b4137-57a0-3ae0-89db-1c33f57ae0c1,LOWN ACQUISITION LLC,134 NORTH ST,NORTH READING,MA,18641315,42.589234,-71.105465,7818262393,0.0,2


In [346]:
print("Organizations Data Quality Profile")
print("=" * 50)

print(f"Rows: {len(organizations):,}")
print(f"Columns: {len(organizations.columns)}")
print(f"Duplicate Rows: {organizations.duplicated().sum():,}")

print("\nMissing Values")
print("=" * 50)

missing = organizations.isna().sum().sort_values(ascending=False)

missing_pct = (
    organizations.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct.round(2),
    "Data Type": organizations.dtypes.astype(str)
})

Organizations Data Quality Profile
Rows: 1,000
Columns: 11
Duplicate Rows: 0

Missing Values


,Missing Count,Missing %,Data Type
Id,0,0.0,str
NAME,0,0.0,str
ADDRESS,0,0.0,str
CITY,0,0.0,str
STATE,0,0.0,str
ZIP,0,0.0,int64
LAT,0,0.0,float64
LON,0,0.0,float64
PHONE,0,0.0,str
REVENUE,0,0.0,float64


In [347]:
print("Organization Validation")
print("=" * 50)

print("Unique Organization IDs:", organizations["Id"].nunique())
print("Duplicate Organization IDs:", organizations["Id"].duplicated().sum())

print("\nOrganization Types")
print("=" * 50)

print(organizations["NAME"].value_counts().head(20))

Organization Validation
Unique Organization IDs: 1000
Duplicate Organization IDs: 0

Organization Types
NAME
BOSTON HEALTH CARE FOR THE HOMELESS PROGRAM INC    8
COMMUNITY HEALTH PROGRAMS INC                      5
GREATER LAWRENCE FAMILY HEALTH CENTER INC          5
MANET COMMUNITY HEALTH CENTER  INC.                5
EDWARD M KENNEDY COMMUNITY HEALTH CENTER INC       5
THE NORTHEAST HEALTH GROUP  INC                    4
HARBOR HEALTH SERVICES INC                         4
NANTUCKET COTTAGE HOSPITAL                         3
WOONSOCKET URGENT CARE PC                          3
BAYADA HOME HEALTH CARE  INC                       3
FAIRVIEW HOSPITAL                                  3
FENWAY COMMUNITY HEALTH CENTER  INC                3
COMMUNITY HEALTH CENTER OF CAPE COD  INC.          3
KINDRED AT HOME                                    3
MARTHA'S VINEYARD HOSPITAL INC                     3
HILLTOWN COMMUNITY HEALTH CENTERS INC              3
METRO WEST REHAB CORP                      

In [348]:
print("Organization Geographic Validation")
print("=" * 50)

print(
    "Invalid latitude:",
    ((organizations["LAT"] < -90) | (organizations["LAT"] > 90)).sum()
)

print(
    "Invalid longitude:",
    ((organizations["LON"] < -180) | (organizations["LON"] > 180)).sum()
)

print(
    "Missing coordinates:",
    organizations[["LAT", "LON"]].isna().any(axis=1).sum()
)

Organization Geographic Validation
Invalid latitude: 0
Invalid longitude: 0
Missing coordinates: 0


In [350]:
print("Organization Columns")
print("=" * 50)

print(organizations.columns.tolist())

Organization Columns
['Id', 'NAME', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'LAT', 'LON', 'PHONE', 'REVENUE', 'UTILIZATION']


In [351]:
organizations.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Id           1000 non-null   str    
 1   NAME         1000 non-null   str    
 2   ADDRESS      1000 non-null   str    
 3   CITY         1000 non-null   str    
 4   STATE        1000 non-null   str    
 5   ZIP          1000 non-null   int64  
 6   LAT          1000 non-null   float64
 7   LON          1000 non-null   float64
 8   PHONE        1000 non-null   str    
 9   REVENUE      1000 non-null   float64
 10  UTILIZATION  1000 non-null   int64  
dtypes: float64(3), int64(2), str(6)
memory usage: 86.1 KB


In [352]:
print("Organization Business Validation")
print("=" * 50)

print("Unique Organization IDs:", organizations["Id"].nunique())
print("Duplicate Organization IDs:", organizations["Id"].duplicated().sum())

print("\nFinancial Validation")
print("=" * 50)

print("Negative revenue:", (organizations["REVENUE"] < 0).sum())
print("Negative utilization:", (organizations["UTILIZATION"] < 0).sum())

print("\nRevenue Summary")
print(organizations["REVENUE"].describe())

print("\nUtilization Summary")
print(organizations["UTILIZATION"].describe())

Organization Business Validation
Unique Organization IDs: 1000
Duplicate Organization IDs: 0

Financial Validation
Negative revenue: 0
Negative utilization: 0

Revenue Summary
count    1000.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: REVENUE, dtype: float64

Utilization Summary
count    1000.000000
mean      472.595000
std      1054.088577
min         1.000000
25%        14.000000
50%       110.000000
75%       415.250000
max      9615.000000
Name: UTILIZATION, dtype: float64


In [353]:
print("Organization Name Validation")
print("=" * 50)

print("Unique names:", organizations["NAME"].nunique())
print(
    "Duplicate names:",
    organizations["NAME"].duplicated().sum()
)

Organization Name Validation
Unique names: 912
Duplicate names: 88


In [354]:
# ============================================================
# Investigate Duplicate Organization Names
# ============================================================

duplicate_names = organizations[
    organizations["NAME"].duplicated(keep=False)
].sort_values("NAME")

print("Duplicate Organization Names")
print("=" * 50)

print(f"Records with repeated names: {len(duplicate_names):,}")
print(f"Unique repeated names: {duplicate_names['NAME'].nunique():,}")

duplicate_names[
    ["Id", "NAME", "CITY", "STATE", "ZIP"]
].head(30)

Duplicate Organization Names
Records with repeated names: 142
Unique repeated names: 54


,Id,NAME,CITY,STATE,ZIP
854,ecd3b321-0a8a-3ce6-81f0-982dd77fce43,ANNA JAQUES HOSPITAL SUBACUTE UNIT,NEWBURYPORT,MA,19503867
647,18a96051-91f4-3cba-af96-056ea14dfee0,ANNA JAQUES HOSPITAL SUBACUTE UNIT,NEWBURYPORT,MA,19503872
637,a525d084-5a8f-3859-a048-2130a4092ec8,BAYADA HOME HEALTH CARE INC,MARLBOROUGH,MA,17521976
603,7c625dc4-2593-31eb-95fd-6a912832b8bb,BAYADA HOME HEALTH CARE INC,DEDHAM,MA,20266704
77,1ddd51d1-7297-3c3b-bad5-98aecd5e4a79,BAYADA HOME HEALTH CARE INC,BURLINGTON,MA,18033735
500,88975fd9-b47e-3af6-af78-c58e1ecf0d5e,BAYSTATE NOBLE HOSPITAL CORPORATION,WESTFIELD,MA,10853628
91,0ddb0329-b59d-3187-874c-602c649249c8,BAYSTATE NOBLE HOSPITAL CORPORATION,WESTFIELD,MA,10853678
490,f43f97f2-af04-3ed0-a42e-3681718cc9b2,BERKSHIRE MEDICAL CENTER INC,PITTSFIELD,MA,12014109
651,1ab2b5f9-cb95-3236-beec-9b9d4a407c61,BERKSHIRE MEDICAL CENTER INC,PITTSFIELD,MA,12014109
793,2f5c26cd-be8d-351a-97af-23fbcaba0967,BETH ISRAEL DEACONESS HOSPITAL PLYMOUTH INC,PLYMOUTH,MA,23607320


In [355]:
organizations_clean = organizations.copy()

In [356]:
print("FINAL ORGANIZATIONS VALIDATION")
print("=" * 55)

print(f"Rows: {len(organizations_clean):,}")
print(f"Columns: {len(organizations_clean.columns)}")
print(f"Unique Organization IDs: {organizations_clean['Id'].nunique():,}")
print(f"Duplicate Rows: {organizations_clean.duplicated().sum():,}")
print(f"Missing Values: {organizations_clean.isna().sum().sum():,}")

print(
    "Invalid Coordinates:",
    (
        (organizations_clean["LAT"] < -90) |
        (organizations_clean["LAT"] > 90) |
        (organizations_clean["LON"] < -180) |
        (organizations_clean["LON"] > 180)
    ).sum()
)

print(
    "Negative Revenue:",
    (organizations_clean["REVENUE"] < 0).sum()
)

print(
    "Negative Utilization:",
    (organizations_clean["UTILIZATION"] < 0).sum()
)

FINAL ORGANIZATIONS VALIDATION
Rows: 1,000
Columns: 11
Unique Organization IDs: 1,000
Duplicate Rows: 0
Missing Values: 0
Invalid Coordinates: 0
Negative Revenue: 0
Negative Utilization: 0


In [357]:
organizations_clean.to_csv(
    PROCESSED_DATA / "organizations_clean.csv",
    index=False
)

print("organizations_clean.csv exported successfully.")
print(
    "File exists:",
    (PROCESSED_DATA / "organizations_clean.csv").exists()
)

organizations_clean.csv exported successfully.
File exists: True


In [358]:
# ============================================================
# Load Payers Dataset
# ============================================================

payers = pd.read_csv(
    RAW_DATA / "payers.csv"
)

print(f"Payers loaded: {len(payers):,}")
payers.head()

Payers loaded: 10


,Id,NAME,OWNERSHIP,ADDRESS,CITY,STATE_HEADQUARTERED,ZIP,PHONE,AMOUNT_COVERED,AMOUNT_UNCOVERED,...,UNCOVERED_ENCOUNTERS,COVERED_MEDICATIONS,UNCOVERED_MEDICATIONS,COVERED_PROCEDURES,UNCOVERED_PROCEDURES,COVERED_IMMUNIZATIONS,UNCOVERED_IMMUNIZATIONS,UNIQUE_CUSTOMERS,QOLS_AVG,MEMBER_MONTHS
0,a735bf55-83e9-331a-899d-a82a60b9f60c,Medicare,GOVERNMENT,NaN,NaN,NaN,NaN,NaN,2.147374e+08,10272056.47,...,0,114902,0,226132,0,15332,0,907,0.755877,159588
1,df166300-5a78-3502-a46a-832842197811,Medicaid,GOVERNMENT,NaN,NaN,NaN,NaN,NaN,3.328398e+08,7407380.99,...,0,34031,0,198675,0,30916,0,1235,0.916398,267936
2,d18ef2e6-ef40-324c-be54-34a5ee865625,Dual Eligible,GOVERNMENT,NaN,NaN,NaN,NaN,NaN,4.640686e+07,851539.39,...,0,16790,0,33133,0,2664,0,198,0.674824,27384
3,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,Humana,PRIVATE,NaN,NaN,NaN,NaN,NaN,1.464394e+08,93728103.60,...,0,31577,0,171613,0,27367,0,649,0.958780,274332
4,b046940f-1664-3047-bca7-dfa76be352a4,Blue Cross Blue Shield,PRIVATE,NaN,NaN,NaN,NaN,NaN,1.045859e+08,30498799.67,...,0,13429,0,63856,0,11471,0,1034,0.414401,221412


In [359]:
print("Payers Data Quality Profile")
print("=" * 50)

print(f"Rows: {len(payers):,}")
print(f"Columns: {len(payers.columns)}")
print(f"Duplicate Rows: {payers.duplicated().sum():,}")

print("\nMissing Values")
print("=" * 50)

missing = payers.isna().sum()

print(
    pd.DataFrame({
        "Missing Count": missing,
        "Missing %": (
            payers.isna().mean() * 100
        ).round(2),
        "Data Type": payers.dtypes.astype(str)
    })
)

print("\nColumns")
print("=" * 50)

print(payers.columns.tolist())

Payers Data Quality Profile
Rows: 10
Columns: 22
Duplicate Rows: 0

Missing Values
                         Missing Count  Missing % Data Type
Id                                   0        0.0       str
NAME                                 0        0.0       str
OWNERSHIP                            0        0.0       str
ADDRESS                             10      100.0   float64
CITY                                10      100.0   float64
STATE_HEADQUARTERED                 10      100.0   float64
ZIP                                 10      100.0   float64
PHONE                               10      100.0   float64
AMOUNT_COVERED                       0        0.0   float64
AMOUNT_UNCOVERED                     0        0.0   float64
REVENUE                              0        0.0   float64
COVERED_ENCOUNTERS                   0        0.0     int64
UNCOVERED_ENCOUNTERS                 0        0.0     int64
COVERED_MEDICATIONS                  0        0.0     int64
UNCOVERED_MEDICAT

In [360]:
print("Payer Business Validation")
print("=" * 50)

print("Unique Payer IDs:", payers["Id"].nunique())
print("Duplicate Payer IDs:", payers["Id"].duplicated().sum())

print("\nPayer Names")
print("=" * 50)
print(payers[["Id", "NAME", "OWNERSHIP"]].to_string(index=False))

print("\nFinancial Validation")
print("=" * 50)

print("Negative Amount Covered:",
      (payers["AMOUNT_COVERED"] < 0).sum())

print("Negative Amount Uncovered:",
      (payers["AMOUNT_UNCOVERED"] < 0).sum())

print("Negative Revenue:",
      (payers["REVENUE"] < 0).sum())

print("\nCoverage Summary")
print("=" * 50)

print("Total Amount Covered:",
      f"${payers['AMOUNT_COVERED'].sum():,.2f}")

print("Total Amount Uncovered:",
      f"${payers['AMOUNT_UNCOVERED'].sum():,.2f}")

print("Total Revenue:",
      f"${payers['REVENUE'].sum():,.2f}")

print("Total Covered Encounters:",
      f"{payers['COVERED_ENCOUNTERS'].sum():,}")

print("Total Uncovered Encounters:",
      f"{payers['UNCOVERED_ENCOUNTERS'].sum():,}")

Payer Business Validation
Unique Payer IDs: 10
Duplicate Payer IDs: 0

Payer Names
                                  Id                   NAME    OWNERSHIP
a735bf55-83e9-331a-899d-a82a60b9f60c               Medicare   GOVERNMENT
df166300-5a78-3502-a46a-832842197811               Medicaid   GOVERNMENT
d18ef2e6-ef40-324c-be54-34a5ee865625          Dual Eligible   GOVERNMENT
26aab0cd-6aba-3e1b-ac5b-05c8867e762c                 Humana      PRIVATE
b046940f-1664-3047-bca7-dfa76be352a4 Blue Cross Blue Shield      PRIVATE
d31fccc3-1767-390d-966a-22a5156f4219       UnitedHealthcare      PRIVATE
0133f751-9229-3cfd-815f-b6d4979bdd6a                  Aetna      PRIVATE
8fa6c185-e44e-3e34-8bd8-39be8694f4ce           Cigna Health      PRIVATE
734afbd6-4794-363b-9bc0-6a3981533ed5                 Anthem      PRIVATE
e03e23c9-4df1-3eb6-a62d-f70f02301496           NO_INSURANCE NO_INSURANCE

Financial Validation
Negative Amount Covered: 0
Negative Amount Uncovered: 0
Negative Revenue: 0

Coverage Summar

In [361]:
print("Payer Utilization Validation")
print("=" * 50)

count_columns = [
    "COVERED_MEDICATIONS",
    "UNCOVERED_MEDICATIONS",
    "COVERED_PROCEDURES",
    "UNCOVERED_PROCEDURES"
]

for col in count_columns:
    print(
        f"{col}:",
        f"{payers[col].sum():,}",
        "| Negative:",
        (payers[col] < 0).sum()
    )

Payer Utilization Validation
COVERED_MEDICATIONS: 237,825 | Negative: 0
UNCOVERED_MEDICATIONS: 35,397 | Negative: 0
COVERED_PROCEDURES: 840,738 | Negative: 0
UNCOVERED_PROCEDURES: 236,598 | Negative: 0


In [362]:
payers_clean = payers.copy()

In [363]:
# ============================================================
# Create and Export Clean Payers Dataset
# ============================================================

payers_clean = payers.copy()

payers_clean.to_csv(
    PROCESSED_DATA / "payers_clean.csv",
    index=False
)

print("payers_clean.csv exported successfully.")

print(
    "File exists:",
    (PROCESSED_DATA / "payers_clean.csv").exists()
)

payers_clean.csv exported successfully.
File exists: True


In [364]:
# ============================================================
# Load Patients Dataset
# ============================================================

patients = pd.read_csv(
    RAW_DATA / "patients.csv"
)

print(f"Patients loaded: {len(patients):,}")
patients.head()

Patients loaded: 2,868


,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,ebde6fa6-7f57-0893-8c60-9bc114b60899,2012-01-18,NaN,999-29-9099,NaN,NaN,NaN,Samuel331,Shanelle862,Bechtelar572,...,Pembroke,Massachusetts,Plymouth County,NaN,0,42.048964,-70.848523,21425.83,18631.94,66105
1,12021186-2496-a178-80b1-5c9e52adf9a3,2018-10-01,NaN,999-62-7777,NaN,NaN,NaN,Elke785,Dolly486,Wisozk929,...,Middleborough,Massachusetts,Plymouth County,NaN,0,41.920544,-70.907976,31659.48,10940.06,139709
2,0b879291-4e25-a926-1253-0604f2f8913f,1973-09-06,NaN,999-77-2848,S99980595,X19589248X,Mr.,Ezequiel972,Mark765,Raynor401,...,Randolph,Massachusetts,Norfolk County,25021.0,2368,42.140430,-71.012097,622427.91,89081.63,118507
3,229514f0-f87a-5e77-2a7b-c0112a2c5801,1992-07-20,NaN,999-23-5414,S99939292,X80307275X,Ms.,Marcelle381,Dean966,Huel628,...,Smith Mills,Massachusetts,Bristol County,NaN,0,41.625507,-71.021901,73344.54,158559.67,31749
4,05219dbb-95b5-7015-3172-52522117ea8a,1994-03-20,NaN,999-30-8427,S99983683,X40068125X,Ms.,Michiko564,Chasity985,Russel238,...,Seekonk,Massachusetts,Bristol County,NaN,0,41.848884,-71.319392,220761.29,605856.72,50017


In [365]:
print("Patients Data Quality Profile")
print("=" * 50)

print(f"Rows: {len(patients):,}")
print(f"Columns: {len(patients.columns):,}")
print(f"Duplicate Rows: {patients.duplicated().sum():,}")

print("\nColumns")
print("=" * 50)

print(patients.columns.tolist())

print("\nMissing Values")
print("=" * 50)

missing = patients.isna().sum()

print(
    pd.DataFrame({
        "Missing Count": missing,
        "Missing %": (patients.isna().mean() * 100).round(2),
        "Data Type": patients.dtypes.astype(str)
    })
)

Patients Data Quality Profile
Rows: 2,868
Columns: 28
Duplicate Rows: 0

Columns
['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME']

Missing Values
                     Missing Count  Missing % Data Type
Id                               0       0.00       str
BIRTHDATE                        0       0.00       str
DEATHDATE                     2500      87.17       str
SSN                              0       0.00       str
DRIVERS                        492      17.15       str
PASSPORT                       629      21.93       str
PREFIX                         566      19.74       str
FIRST                            0       0.00       str
MIDDLE                         574      20.01       str
LAST                             0       0.0

In [366]:
print("Complete Patient Schema")
print("=" * 60)

for i, column in enumerate(patients.columns, start=1):
    print(f"{i:2}. {column}")

Complete Patient Schema
 1. Id
 2. BIRTHDATE
 3. DEATHDATE
 4. SSN
 5. DRIVERS
 6. PASSPORT
 7. PREFIX
 8. FIRST
 9. MIDDLE
10. LAST
11. SUFFIX
12. MAIDEN
13. MARITAL
14. RACE
15. ETHNICITY
16. GENDER
17. BIRTHPLACE
18. ADDRESS
19. CITY
20. STATE
21. COUNTY
22. FIPS
23. ZIP
24. LAT
25. LON
26. HEALTHCARE_EXPENSES
27. HEALTHCARE_COVERAGE
28. INCOME


In [367]:
# ============================================================
# Create Analytical Patients Dataset
# ============================================================

patient_columns = [
    "Id",
    "BIRTHDATE",
    "DEATHDATE",
    "MARITAL",
    "RACE",
    "ETHNICITY",
    "GENDER",
    "CITY",
    "STATE",
    "COUNTY",
    "FIPS",
    "ZIP",
    "LON",
    "HEALTHCARE_EXPENSES",
    "HEALTHCARE_COVERAGE",
    "INCOME"
]

patients_clean = patients[patient_columns].copy()

print("Patients analytical dataset created.")
print(f"Rows: {len(patients_clean):,}")
print(f"Columns: {len(patients_clean.columns)}")

Patients analytical dataset created.
Rows: 2,868
Columns: 16


In [368]:
patients_clean["BIRTHDATE"] = pd.to_datetime(
    patients_clean["BIRTHDATE"],
    errors="coerce"
)

patients_clean["DEATHDATE"] = pd.to_datetime(
    patients_clean["DEATHDATE"],
    errors="coerce"
)

print("Birthdate range:")
print(patients_clean["BIRTHDATE"].min())
print(patients_clean["BIRTHDATE"].max())

print("\nDeathdate range:")
print(patients_clean["DEATHDATE"].min())
print(patients_clean["DEATHDATE"].max())

Birthdate range:
1916-04-04 00:00:00
2026-08-03 00:00:00

Deathdate range:
1932-01-03 00:00:00
2026-06-18 00:00:00


In [369]:
# ============================================================
# Patient Analytical Attributes
# ============================================================

analysis_date = pd.Timestamp("2022-12-31")

patients_clean["AGE_2022"] = (
    (analysis_date - patients_clean["BIRTHDATE"]).dt.days / 365.25
).astype(int)

patients_clean["IS_DECEASED"] = (
    patients_clean["DEATHDATE"].notna()
)

patients_clean["BIRTH_YEAR"] = (
    patients_clean["BIRTHDATE"].dt.year
)

print(
    patients_clean[
        [
            "Id",
            "BIRTHDATE",
            "AGE_2022",
            "IS_DECEASED",
            "BIRTH_YEAR"
        ]
    ].head()
)

                                     Id  BIRTHDATE  AGE_2022  IS_DECEASED  \
0  ebde6fa6-7f57-0893-8c60-9bc114b60899 2012-01-18        10        False   
1  12021186-2496-a178-80b1-5c9e52adf9a3 2018-10-01         4        False   
2  0b879291-4e25-a926-1253-0604f2f8913f 1973-09-06        49        False   
3  229514f0-f87a-5e77-2a7b-c0112a2c5801 1992-07-20        30        False   
4  05219dbb-95b5-7015-3172-52522117ea8a 1994-03-20        28        False   

   BIRTH_YEAR  
0        2012  
1        2018  
2        1973  
3        1992  
4        1994  


In [370]:
print("PATIENTS VALIDATION")
print("=" * 55)

print(f"Rows: {len(patients_clean):,}")
print(f"Columns: {len(patients_clean.columns)}")

print(
    "Unique Patient IDs:",
    f"{patients_clean['Id'].nunique():,}"
)

print(
    "Duplicate rows:",
    patients_clean.duplicated().sum()
)

print(
    "Duplicate Patient IDs:",
    patients_clean["Id"].duplicated().sum()
)

print(
    "Missing Patient IDs:",
    patients_clean["Id"].isna().sum()
)

print(
    "Negative ages:",
    (patients_clean["AGE_2022"] < 0).sum()
)

print(
    "Deceased patients:",
    patients_clean["IS_DECEASED"].sum()
)

PATIENTS VALIDATION
Rows: 2,868
Columns: 19
Unique Patient IDs: 2,868
Duplicate rows: 0
Duplicate Patient IDs: 0
Missing Patient IDs: 0
Negative ages: 66
Deceased patients: 368


In [371]:
print("Age Validation")
print("=" * 50)

print(patients_clean["AGE_2022"].describe())

print(
    "\nPatients under 0:",
    (patients_clean["AGE_2022"] < 0).sum()
)

print(
    "Patients over 120:",
    (patients_clean["AGE_2022"] > 120).sum()
)

Age Validation
count    2868.000000
mean       40.883891
std        26.158117
min        -3.000000
25%        20.000000
50%        41.000000
75%        59.000000
max       106.000000
Name: AGE_2022, dtype: float64

Patients under 0: 66
Patients over 120: 0


In [372]:
invalid_age = patients_clean[
    patients_clean["AGE_2022"] < 0
].copy()

print("Patients with invalid age")
print("=" * 50)

print(
    invalid_age[
        ["Id", "BIRTHDATE", "DEATHDATE", "AGE_2022"]
    ].sort_values("BIRTHDATE").to_string(index=False)
)

Patients with invalid age
                                  Id  BIRTHDATE  DEATHDATE  AGE_2022
0c22542b-0701-d66d-5497-e3d2d13a7748 2024-02-16        NaT        -1
bf6d3077-fdcf-7bbe-4dbb-a07e1919e217 2024-02-27        NaT        -1
8e1163fc-b5fe-ce4d-8513-e40196cb8ff6 2024-03-26        NaT        -1
8605ba5b-56b6-d330-f8d3-e60378c4f2da 2024-04-04        NaT        -1
e59d601b-9865-8f56-a959-275d33f29c97 2024-04-20        NaT        -1
b119b61b-29fd-5472-0f47-4cde5634f3c7 2024-05-03 2025-10-06        -1
d8b1a4f7-fbb5-3b29-c22a-e1cb7300fabd 2024-05-03        NaT        -1
675820d9-7882-304a-5a39-ae695c20d3a1 2024-05-10        NaT        -1
787fd681-d6c8-2ca7-0b9a-c5314c3389e1 2024-05-28        NaT        -1
cde38177-5b2a-eea2-377c-484ad2cf413a 2024-06-26        NaT        -1
a83ed2f7-fb91-ebaa-fb8b-38275640dd8c 2024-07-21        NaT        -1
38cca300-43ce-34ec-7acd-640975d52ccd 2024-07-31        NaT        -1
406a524a-87fd-6f6a-ac51-85c78f71571b 2024-10-17        NaT        -1
1e7a12ad

In [373]:
patients_clean.loc[
    patients_clean["AGE_2022"] < 0,
    "AGE_2022"
] = pd.NA

In [374]:
print("Final Age Validation")
print("=" * 50)

print(
    "Missing AGE_2022:",
    patients_clean["AGE_2022"].isna().sum()
)

print(
    "Negative AGE_2022:",
    (patients_clean["AGE_2022"].dropna() < 0).sum()
)

print(
    "Age > 120:",
    (patients_clean["AGE_2022"].dropna() > 120).sum()
)

Final Age Validation
Missing AGE_2022: 66
Negative AGE_2022: 0
Age > 120: 0


In [375]:
print("FINAL PATIENTS VALIDATION")
print("=" * 55)

print(f"Rows: {len(patients_clean):,}")
print(f"Columns: {len(patients_clean.columns)}")

print(
    "Unique Patient IDs:",
    f"{patients_clean['Id'].nunique():,}"
)

print(
    "Duplicate Patient IDs:",
    patients_clean["Id"].duplicated().sum()
)

print(
    "Duplicate Rows:",
    patients_clean.duplicated().sum()
)

print(
    "Missing Patient IDs:",
    patients_clean["Id"].isna().sum()
)

print(
    "Invalid Ages:",
    (
        (patients_clean["AGE_2022"].dropna() < 0) |
        (patients_clean["AGE_2022"].dropna() > 120)
    ).sum()
)

FINAL PATIENTS VALIDATION
Rows: 2,868
Columns: 19
Unique Patient IDs: 2,868
Duplicate Patient IDs: 0
Duplicate Rows: 0
Missing Patient IDs: 0
Invalid Ages: 0


In [376]:
patients_clean.to_csv(
    PROCESSED_DATA / "patients_clean.csv",
    index=False
)

print("patients_clean.csv exported successfully.")

print(
    "File exists:",
    (PROCESSED_DATA / "patients_clean.csv").exists()
)

patients_clean.csv exported successfully.
File exists: True


In [377]:
# ============================================================
# Load Encounters Dataset
# ============================================================

encounters = pd.read_csv(
    RAW_DATA / "encounters.csv"
)

print(f"Encounters loaded: {len(encounters):,}")
encounters.head()

Encounters loaded: 169,640


,Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
0,ebde6fa6-7f57-0893-748d-6eeeccfa2731,2013-09-28T22:10:35Z,2013-09-28T22:25:35Z,ebde6fa6-7f57-0893-8c60-9bc114b60899,6aae7a31-90df-3455-ad8d-81f8cf2d21e8,b246e387-68c1-36e8-9991-9be11f671291,734afbd6-4794-363b-9bc0-6a3981533ed5,ambulatory,185345009,Encounter for symptom (procedure),85.55,92.93,74.34,75498004.0,Acute bacterial sinusitis (disorder)
1,12021186-2496-a178-c232-3afa8dcfd265,2018-10-01T14:53:56Z,2018-10-01T15:08:56Z,12021186-2496-a178-80b1-5c9e52adf9a3,b574fc4f-313c-38a4-9baf-7e12e431a499,44f18836-01b5-3101-beca-390e3e1bb350,b046940f-1664-3047-bca7-dfa76be352a4,wellness,410620009,Well child visit (procedure),136.80,347.38,0.00,NaN,NaN
2,0b879291-4e25-a926-4097-793c8006d6bc,1991-10-31T14:16:01Z,1991-10-31T14:58:35Z,0b879291-4e25-a926-1253-0604f2f8913f,31e00f85-6909-3d9f-afc6-5ae507ab0c06,13b552ae-997c-33c8-9f0f-88ac322ba107,e03e23c9-4df1-3eb6-a62d-f70f02301496,wellness,162673000,General examination of patient (procedure),136.80,704.20,0.00,NaN,NaN
3,0b879291-4e25-a926-3421-aadceefbf206,2001-11-15T14:16:01Z,2001-11-15T14:46:03Z,0b879291-4e25-a926-1253-0604f2f8913f,31e00f85-6909-3d9f-afc6-5ae507ab0c06,13b552ae-997c-33c8-9f0f-88ac322ba107,e03e23c9-4df1-3eb6-a62d-f70f02301496,wellness,162673000,General examination of patient (procedure),136.80,919.90,0.00,NaN,NaN
4,229514f0-f87a-5e77-d028-ea97e7a67de9,2010-09-13T06:07:54Z,2010-09-13T06:59:44Z,229514f0-f87a-5e77-2a7b-c0112a2c5801,3f12ebb4-e03c-3453-88d2-4fc9682383df,253bf219-7089-3dda-bafd-adc878fcaa6b,0133f751-9229-3cfd-815f-b6d4979bdd6a,wellness,162673000,General examination of patient (procedure),136.80,704.20,0.00,NaN,NaN


In [378]:
print("Encounters Data Quality Profile")
print("=" * 55)

print(f"Rows: {len(encounters):,}")
print(f"Columns: {len(encounters.columns)}")
print(f"Duplicate Rows: {encounters.duplicated().sum():,}")

print("\nColumns")
print("=" * 55)

print(encounters.columns.tolist())

print("\nMissing Values")
print("=" * 55)

missing = encounters.isna().sum()

print(
    pd.DataFrame({
        "Missing Count": missing,
        "Missing %": (encounters.isna().mean() * 100).round(2),
        "Data Type": encounters.dtypes.astype(str)
    })
)

Encounters Data Quality Profile
Rows: 169,640
Columns: 15
Duplicate Rows: 0

Columns
['Id', 'START', 'STOP', 'PATIENT', 'ORGANIZATION', 'PROVIDER', 'PAYER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE', 'REASONDESCRIPTION']

Missing Values
                     Missing Count  Missing % Data Type
Id                               0       0.00       str
START                            0       0.00       str
STOP                             0       0.00       str
PATIENT                          0       0.00       str
ORGANIZATION                     0       0.00       str
PROVIDER                         0       0.00       str
PAYER                            0       0.00       str
ENCOUNTERCLASS                   0       0.00       str
CODE                             0       0.00     int64
DESCRIPTION                      0       0.00       str
BASE_ENCOUNTER_COST              0       0.00   float64
TOTAL_CLAIM_COST  

In [379]:
# ============================================================
# Encounter Relationship Validation
# ============================================================

print("ENCOUNTER RELATIONSHIP VALIDATION")
print("=" * 55)

print("Unique Encounter IDs:",
      encounters["Id"].nunique())

print("Duplicate Encounter IDs:",
      encounters["Id"].duplicated().sum())

print("Unique Patients in Encounters:",
      encounters["PATIENT"].nunique())

print("Unique Organizations:",
      encounters["ORGANIZATION"].nunique())

print("Unique Providers:",
      encounters["PROVIDER"].nunique())

print("Unique Payers:",
      encounters["PAYER"].nunique())

print("\nReference Dataset Counts")
print("=" * 55)

print("Patients:", patients_clean["Id"].nunique())
print("Organizations:", organizations_clean["Id"].nunique())
print("Providers:", providers_clean["Id"].nunique())
print("Payers:", payers_clean["Id"].nunique())

ENCOUNTER RELATIONSHIP VALIDATION
Unique Encounter IDs: 169640
Duplicate Encounter IDs: 0
Unique Patients in Encounters: 2868
Unique Organizations: 908
Unique Providers: 908
Unique Payers: 10

Reference Dataset Counts
Patients: 2868
Organizations: 1000
Providers: 1000
Payers: 10


In [380]:
print("Foreign Key Validation")
print("=" * 55)

print(
    "Patients not found:",
    (~encounters["PATIENT"].isin(patients_clean["Id"])).sum()
)

print(
    "Organizations not found:",
    (~encounters["ORGANIZATION"].isin(organizations_clean["Id"])).sum()
)

print(
    "Providers not found:",
    (~encounters["PROVIDER"].isin(providers_clean["Id"])).sum()
)

print(
    "Payers not found:",
    (~encounters["PAYER"].isin(payers_clean["Id"])).sum()
)

Foreign Key Validation
Patients not found: 0
Organizations not found: 0
Providers not found: 0
Payers not found: 0


In [381]:
# ============================================================
# Encounter Date Validation
# ============================================================

encounters["START"] = pd.to_datetime(
    encounters["START"],
    errors="coerce"
)

encounters["STOP"] = pd.to_datetime(
    encounters["STOP"],
    errors="coerce"
)

negative_duration = (
    encounters["STOP"] < encounters["START"]
).sum()

zero_duration = (
    encounters["STOP"] == encounters["START"]
).sum()

print("Encounter Duration Validation")
print("=" * 55)

print("Negative durations:", negative_duration)
print("Zero durations:", zero_duration)
print("Missing START:", encounters["START"].isna().sum())
print("Missing STOP:", encounters["STOP"].isna().sum())

Encounter Duration Validation
Negative durations: 0
Zero durations: 0
Missing START: 0
Missing STOP: 0


In [382]:
print("Foreign Key Validation")
print("=" * 55)

print(
    "Patients not found:",
    (~encounters["PATIENT"].isin(patients_clean["Id"])).sum()
)

print(
    "Organizations not found:",
    (~encounters["ORGANIZATION"].isin(organizations_clean["Id"])).sum()
)

print(
    "Providers not found:",
    (~encounters["PROVIDER"].isin(providers_clean["Id"])).sum()
)

print(
    "Payers not found:",
    (~encounters["PAYER"].isin(payers_clean["Id"])).sum()
)

Foreign Key Validation
Patients not found: 0
Organizations not found: 0
Providers not found: 0
Payers not found: 0


In [383]:
print("Encounter Cost Validation")
print("=" * 55)

print(
    "Negative base encounter costs:",
    (encounters["BASE_ENCOUNTER_COST"] < 0).sum()
)

print(
    "Negative total claim costs:",
    (encounters["TOTAL_CLAIM_COST"] < 0).sum()
)

print(
    "Negative payer coverage:",
    (encounters["PAYER_COVERAGE"] < 0).sum()
)

print("\nCost Summary")
print("=" * 55)

print(
    encounters[
        [
            "BASE_ENCOUNTER_COST",
            "TOTAL_CLAIM_COST",
            "PAYER_COVERAGE"
        ]
    ].describe()
)

Encounter Cost Validation
Negative base encounter costs: 0
Negative total claim costs: 0
Negative payer coverage: 0

Cost Summary
       BASE_ENCOUNTER_COST  TOTAL_CLAIM_COST  PAYER_COVERAGE
count        169640.000000     169640.000000   169640.000000
mean            112.373889       2715.965287     1973.354049
std              27.436532       6976.036205     5902.253250
min              75.000000         75.000000        0.000000
25%              85.550000        516.950000        0.000000
50%              87.710000        914.780000      485.870000
75%             142.580000       1871.780000     1303.700000
max             146.180000     244426.550000   207416.210000


In [384]:
print("Encounter Class Distribution")
print("=" * 55)

print(
    encounters["ENCOUNTERCLASS"]
    .value_counts()
)

Encounter Class Distribution
ENCOUNTERCLASS
ambulatory    94578
wellness      35021
outpatient    21317
urgentcare     7243
emergency      6805
inpatient      2711
home            681
snf             451
virtual         429
hospice         404
Name: count, dtype: int64


In [385]:
# ============================================================
# Encounter Duration
# ============================================================

encounters["EncounterDurationHours"] = (
    encounters["STOP"] - encounters["START"]
).dt.total_seconds() / 3600

print("Encounter Duration Summary")
print("=" * 55)

print(encounters["EncounterDurationHours"].describe())

Encounter Duration Summary
count    169640.000000
mean          5.681472
std          48.778552
min           0.250000
25%           0.250000
50%           0.758889
75%           2.450000
max        2760.250000
Name: EncounterDurationHours, dtype: float64


In [386]:
print("Encounter Date Range")
print("=" * 55)

print("Earliest:", encounters["START"].min())
print("Latest:", encounters["START"].max())

Encounter Date Range
Earliest: 1920-01-02 00:24:20+00:00
Latest: 2026-08-09 15:20:14+00:00


In [387]:
# ============================================================
# 2020-2022 Analytical Population
# ============================================================

analysis_start = pd.Timestamp("2020-01-01", tz="UTC")
analysis_end = pd.Timestamp("2022-12-31 23:59:59", tz="UTC")

encounters_2020_2022 = encounters[
    (encounters["START"] >= analysis_start) &
    (encounters["START"] <= analysis_end)
].copy()

print("Encounters 2020-2022")
print("=" * 55)

print(f"Records: {len(encounters_2020_2022):,}")
print(
    f"Unique patients: "
    f"{encounters_2020_2022['PATIENT'].nunique():,}"
)
print(
    f"Unique encounters: "
    f"{encounters_2020_2022['Id'].nunique():,}"
)

Encounters 2020-2022
Records: 37,289
Unique patients: 2,493
Unique encounters: 37,289


In [388]:
encounters["EncounterDurationHours"] = (
    encounters["STOP"] - encounters["START"]
).dt.total_seconds() / 3600

print("Encounter Duration Summary")
print("=" * 55)

print(encounters["EncounterDurationHours"].describe())

Encounter Duration Summary
count    169640.000000
mean          5.681472
std          48.778552
min           0.250000
25%           0.250000
50%           0.758889
75%           2.450000
max        2760.250000
Name: EncounterDurationHours, dtype: float64


In [389]:
print("Encounter Date Range")
print("=" * 55)

print("Earliest:", encounters["START"].min())
print("Latest:", encounters["START"].max())

Encounter Date Range
Earliest: 1920-01-02 00:24:20+00:00
Latest: 2026-08-09 15:20:14+00:00


In [390]:
analysis_start = pd.Timestamp("2020-01-01", tz="UTC")
analysis_end = pd.Timestamp("2022-12-31 23:59:59", tz="UTC")

encounters_2020_2022 = encounters[
    (encounters["START"] >= analysis_start) &
    (encounters["START"] <= analysis_end)
].copy()

print("Encounters 2020-2022")
print("=" * 55)

print(f"Records: {len(encounters_2020_2022):,}")
print(f"Unique patients: {encounters_2020_2022['PATIENT'].nunique():,}")
print(f"Unique encounters: {encounters_2020_2022['Id'].nunique():,}")

Encounters 2020-2022
Records: 37,289
Unique patients: 2,493
Unique encounters: 37,289


In [391]:
print("2020-2022 ENCOUNTERS VALIDATION")
print("=" * 55)

print(f"Rows: {len(encounters_2020_2022):,}")
print(f"Unique patients: {encounters_2020_2022['PATIENT'].nunique():,}")
print(f"Unique encounters: {encounters_2020_2022['Id'].nunique():,}")

print(
    "Duplicate rows:",
    encounters_2020_2022.duplicated().sum()
)

print(
    "Duplicate encounter IDs:",
    encounters_2020_2022["Id"].duplicated().sum()
)

print(
    "Missing patient IDs:",
    encounters_2020_2022["PATIENT"].isna().sum()
)

print(
    "Missing encounter classes:",
    encounters_2020_2022["ENCOUNTERCLASS"].isna().sum()
)

print(
    "Negative durations:",
    (encounters_2020_2022["EncounterDurationHours"] < 0).sum()
)

2020-2022 ENCOUNTERS VALIDATION
Rows: 37,289
Unique patients: 2,493
Unique encounters: 37,289
Duplicate rows: 0
Duplicate encounter IDs: 0
Missing patient IDs: 0
Missing encounter classes: 0
Negative durations: 0


In [392]:
print("2020-2022 COST VALIDATION")
print("=" * 55)

print(
    "Negative base costs:",
    (encounters_2020_2022["BASE_ENCOUNTER_COST"] < 0).sum()
)

print(
    "Negative claim costs:",
    (encounters_2020_2022["TOTAL_CLAIM_COST"] < 0).sum()
)

print(
    "Negative payer coverage:",
    (encounters_2020_2022["PAYER_COVERAGE"] < 0).sum()
)

2020-2022 COST VALIDATION
Negative base costs: 0
Negative claim costs: 0
Negative payer coverage: 0


In [393]:
print("FINAL ENCOUNTERS VALIDATION")
print("=" * 55)

print(f"Rows: {len(encounters_2020_2022):,}")
print(f"Unique patients: {encounters_2020_2022['PATIENT'].nunique():,}")
print(f"Duplicate rows: {encounters_2020_2022.duplicated().sum():,}")
print(f"Duplicate encounter IDs: {encounters_2020_2022['Id'].duplicated().sum():,}")

print(
    "Missing PATIENT:",
    encounters_2020_2022["PATIENT"].isna().sum()
)

print(
    "Missing DESCRIPTION:",
    encounters_2020_2022["DESCRIPTION"].isna().sum()
)

print(
    "Missing ENCOUNTERCLASS:",
    encounters_2020_2022["ENCOUNTERCLASS"].isna().sum()
)

print(
    "Negative durations:",
    (encounters_2020_2022["EncounterDurationHours"] < 0).sum()
)

FINAL ENCOUNTERS VALIDATION
Rows: 37,289
Unique patients: 2,493
Duplicate rows: 0
Duplicate encounter IDs: 0
Missing PATIENT: 0
Missing DESCRIPTION: 0
Missing ENCOUNTERCLASS: 0
Negative durations: 0


In [394]:
encounters_clean = encounters_2020_2022.copy()

encounters_clean.to_csv(
    PROCESSED_DATA / "encounters_clean.csv",
    index=False
)

print("encounters_clean.csv exported successfully.")

print(
    "File exists:",
    (PROCESSED_DATA / "encounters_clean.csv").exists()
)

encounters_clean.csv exported successfully.
File exists: True


In [395]:
encounters_clean = encounters_2020_2022.copy()

encounters_clean.to_csv(
    PROCESSED_DATA / "encounters_clean.csv",
    index=False
)

print("encounters_clean.csv exported successfully.")

print(
    "File exists:",
    (PROCESSED_DATA / "encounters_clean.csv").exists()
)

encounters_clean.csv exported successfully.
File exists: True
